In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import json
import joblib

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

BINARY_BUNDLE = (
    PROJECT_ROOT / "models/binary/xgboost_binary_bundle.joblib"
)

MULTICLASS_DIR = PROJECT_ROOT / "models/multiclass"

print("Binary bundle exists:", BINARY_BUNDLE.exists())
print("\nMulticlass model files:")

for path in sorted(MULTICLASS_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")

Binary bundle exists: True

Multiclass model files:
multiclass_label_mapping.json (768 bytes)
xgboost_multiclass_bundle.joblib (6,374,809 bytes)
xgboost_multiclass_model.json (8,201,125 bytes)


In [3]:
from pathlib import Path
import joblib
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

BINARY_BUNDLE = (
    PROJECT_ROOT / "models/binary/xgboost_binary_bundle.joblib"
)
MULTICLASS_BUNDLE = (
    PROJECT_ROOT / "models/multiclass/xgboost_multiclass_bundle.joblib"
)
LABEL_MAPPING_FILE = (
    PROJECT_ROOT / "models/multiclass/multiclass_label_mapping.json"
)

assert BINARY_BUNDLE.exists(), "Binary bundle not found"
assert MULTICLASS_BUNDLE.exists(), "Multiclass bundle not found"
assert LABEL_MAPPING_FILE.exists(), "Label mapping file not found"

binary_bundle = joblib.load(BINARY_BUNDLE)
multiclass_bundle = joblib.load(MULTICLASS_BUNDLE)

with open(LABEL_MAPPING_FILE, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

def inspect_bundle(name, bundle):
    print(f"\n{name}")
    print("=" * 50)
    print("Object type:", type(bundle).__name__)

    if isinstance(bundle, dict):
        print("Keys:", list(bundle.keys()))

        for key, value in bundle.items():
            if hasattr(value, "predict"):
                print(f"{key}: model object ({type(value).__name__})")
            elif isinstance(value, (list, tuple)):
                print(f"{key}: sequence, length={len(value)}")
            else:
                print(f"{key}: {type(value).__name__}")
    else:
        print("Has predict:", hasattr(bundle, "predict"))
        print("Has predict_proba:", hasattr(bundle, "predict_proba"))

inspect_bundle("BINARY BUNDLE", binary_bundle)
inspect_bundle("MULTICLASS BUNDLE", multiclass_bundle)

print("\nLABEL MAPPING")
print("=" * 50)
print(label_mapping)

/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:16:32] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:16:32] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:16:32] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:16:34] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:16:34] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: U


BINARY BUNDLE
Object type: dict
Keys: ['model', 'imputer', 'feature_columns', 'label_mapping', 'decision_threshold', 'best_iteration', 'validation_metrics', 'created_utc', 'xgboost_version']
model: model object (XGBClassifier)
imputer: SimpleImputer
feature_columns: sequence, length=78
label_mapping: dict
decision_threshold: float
best_iteration: int
validation_metrics: dict
created_utc: str
xgboost_version: str

MULTICLASS BUNDLE
Object type: dict
Keys: ['model', 'imputer', 'feature_columns', 'class_names', 'class_to_id', 'id_to_class', 'best_iteration', 'num_classes', 'model_type']
model: model object (XGBClassifier)
imputer: SimpleImputer
feature_columns: sequence, length=78
class_names: sequence, length=15
class_to_id: dict
id_to_class: dict
best_iteration: int
num_classes: int
model_type: str

LABEL MAPPING
{'class_to_id': {'BENIGN': 0, 'Bot': 1, 'DDoS': 2, 'DoS GoldenEye': 3, 'DoS Hulk': 4, 'DoS Slowhttptest': 5, 'DoS slowloris': 6, 'FTP-Patator': 7, 'Heartbleed': 8, 'Infiltrati

In [ ]:

import numpy as np
import pandas as pd

# 1. Retrieve objects from the saved bundles
binary_model = binary_bundle["model"]
binary_imputer = binary_bundle["imputer"]
binary_features = list(binary_bundle["feature_columns"])
binary_threshold = float(binary_bundle["decision_threshold"])

multiclass_model = multiclass_bundle["model"]
multiclass_imputer = multiclass_bundle["imputer"]
multiclass_features = list(multiclass_bundle["feature_columns"])

id_to_class = {
    int(class_id): class_name
    for class_id, class_name in label_mapping["id_to_class"].items()
}

# 2. Check feature and class compatibility
assert len(binary_features) == 78
assert len(multiclass_features) == 78
assert binary_features == multiclass_features, (
    "Feature names or order differ between the models."
)
assert len(id_to_class) == 15
assert binary_bundle["label_mapping"] == {
    "BENIGN": 0,
    "ATTACK": 1
}, "Unexpected binary label mapping."

print("Feature count:", len(binary_features))
print("Feature order matches:", binary_features == multiclass_features)
print("Binary threshold:", binary_threshold)
print("Multiclass classes:", len(id_to_class))

# 3. Load a small sample from the test split
PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
TEST_FILE = PROJECT_ROOT / "data/splits/test.parquet"

test_sample = pd.read_parquet(TEST_FILE).head(1000)

# Ensure all required features exist and are in model order
missing_features = set(binary_features) - set(test_sample.columns)
assert not missing_features, f"Missing features: {missing_features}"

X_sample = test_sample[binary_features].copy()
X_sample = X_sample.replace([np.inf, -np.inf], np.nan)

# 4. Impute using each model's saved imputer
X_binary = binary_imputer.transform(X_sample)
X_multiclass = multiclass_imputer.transform(X_sample)

assert np.isfinite(X_binary).all()
assert np.isfinite(X_multiclass).all()

# 5. Binary prediction
binary_probabilities = binary_model.predict_proba(X_binary)[:, 1]
binary_predictions = (
    binary_probabilities >= binary_threshold
).astype(np.int8)

# 6. Multiclass prediction only for records classified as attacks
hybrid_predictions = np.zeros(len(X_sample), dtype=np.int64)
attack_mask = binary_predictions == 1

if attack_mask.any():
    attack_class_ids = multiclass_model.predict(
        X_multiclass[attack_mask]
    ).astype(np.int64)

    hybrid_predictions[attack_mask] = attack_class_ids

# 7. Convert IDs into readable labels
predicted_labels = [
    id_to_class[int(class_id)]
    for class_id in hybrid_predictions
]

hybrid_sample_results = pd.DataFrame({
    "binary_attack_probability": binary_probabilities,
    "binary_prediction": np.where(
        attack_mask, "ATTACK", "BENIGN"
    ),
    "hybrid_prediction_id": hybrid_predictions,
    "hybrid_prediction": predicted_labels
})

print("\nHYBRID INFERENCE CHECK")
print("=" * 45)
print("Sample records:", len(test_sample))
print("Routed to multiclass model:", int(attack_mask.sum()))
print("Predicted benign:", int((~attack_mask).sum()))
print("Missing prediction values:",
      int(hybrid_sample_results.isna().sum().sum()))

display(hybrid_sample_results.head(10))


Feature count: 78
Feature order matches: True
Binary threshold: 0.5
Multiclass classes: 15

HYBRID INFERENCE CHECK
Sample records: 1000
Routed to multiclass model: 0
Predicted benign: 1000
Missing prediction values: 0


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(


,binary_attack_probability,binary_prediction,hybrid_prediction_id,hybrid_prediction
0,3.573966e-06,BENIGN,0,BENIGN
1,8.288955e-07,BENIGN,0,BENIGN
2,3.406847e-07,BENIGN,0,BENIGN
3,3.578696e-07,BENIGN,0,BENIGN
4,2.155481e-05,BENIGN,0,BENIGN
5,2.604047e-06,BENIGN,0,BENIGN
6,2.604047e-06,BENIGN,0,BENIGN
7,8.217550e-07,BENIGN,0,BENIGN
8,1.206425e-05,BENIGN,0,BENIGN
9,1.838673e-06,BENIGN,0,BENIGN


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
TEST_FILE = PROJECT_ROOT / "data" / "splits" / "test.parquet"

assert TEST_FILE.exists(), f"Test file not found: {TEST_FILE}"

print("Test file found:", TEST_FILE)
print("File size:", f"{TEST_FILE.stat().st_size / (1024**2):.2f} MB")


Test file found: /content/drive/MyDrive/H-IDS/data/splits/test.parquet
File size: 74.84 MB


In [ ]:
# Load the decision threshold from the saved binary model bundle
binary_threshold = float(
    binary_bundle["decision_threshold"]
)

print("Binary decision threshold:", binary_threshold)

assert 0.0 <= binary_threshold <= 1.0
print("Threshold check passed.")

Binary decision threshold: 0.5
Threshold check passed.


In [ ]:

import pandas as pd
import numpy as np

# 1. Validate the feature configuration
assert len(binary_features) == 78, (
    f"Expected 78 binary features, got {len(binary_features)}"
)

assert len(multiclass_bundle["feature_columns"]) == 78

assert list(binary_features) == list(
    binary_bundle["feature_columns"]
), "Binary feature order does not match the saved bundle."

assert list(binary_features) == list(
    multiclass_bundle["feature_columns"]
), "Multiclass feature order does not match the saved bundle."

# 2. Load only the required test columns
test_df = pd.read_parquet(
    TEST_FILE,
    columns=list(binary_features) + ["label"]
)

assert not test_df["label"].isna().any(), (
    "Missing true labels found in test data."
)

# 3. Draw a reproducible sample
sample_df = test_df.sample(
    n=min(5000, len(test_df)),
    random_state=42
).reset_index(drop=True)

X_sample = sample_df[binary_features].copy()

# Normalize infinities and convert features to numeric values
X_sample = X_sample.replace(
    [np.inf, -np.inf], np.nan
)

# 4. Apply the fitted imputers
X_binary = binary_imputer.transform(X_sample)
X_multiclass = multiclass_imputer.transform(X_sample)

assert np.isfinite(X_binary).all()
assert np.isfinite(X_multiclass).all()

# 5. Binary prediction
binary_probabilities = binary_model.predict_proba(
    X_binary
)[:, 1]

assert np.isfinite(binary_probabilities).all()

binary_predictions = (
    binary_probabilities >= binary_threshold
)

attack_mask = binary_predictions

# 6. Hybrid prediction
# Class ID 0 represents BENIGN.
hybrid_predictions = np.zeros(
    len(sample_df), dtype=np.int64
)

if attack_mask.any():
    hybrid_predictions[attack_mask] = (
        multiclass_model.predict(
            X_multiclass[attack_mask]
        ).astype(np.int64)
    )

# 7. Validate multiclass output IDs
assert np.isin(
    hybrid_predictions,
    list(map(int, id_to_class.keys()))
).all(), "Unexpected predicted class ID."

predicted_labels = [
    id_to_class[int(class_id)]
    for class_id in hybrid_predictions
]

# 8. Create diagnostic results
results = pd.DataFrame({
    "true_label": sample_df["label"].to_numpy(),
    "binary_probability": binary_probabilities,
    "binary_prediction": np.where(
        attack_mask, "ATTACK", "BENIGN"
    ),
    "hybrid_prediction": predicted_labels
})

print("SAMPLE INFERENCE CHECKS PASSED")
print("=" * 50)
print("Sample size:", len(results))
print("Records routed to multiclass:", int(attack_mask.sum()))
print("Records classified as benign:", int((~attack_mask).sum()))

print("\nTRUE LABEL DISTRIBUTION")
print(results["true_label"].value_counts())

print("\nBINARY PREDICTION COUNTS")
print(results["binary_prediction"].value_counts())

print("\nHYBRID PREDICTION DISTRIBUTION")
print(results["hybrid_prediction"].value_counts())

print("\nSAMPLE PREDICTIONS")
display(results.sample(
    min(15, len(results)),
    random_state=7
))

del test_df, X_sample, X_binary, X_multiclass


SAMPLE INFERENCE CHECKS PASSED
Sample size: 5000
Records routed to multiclass: 922
Records classified as benign: 4078

TRUE LABEL DISTRIBUTION
true_label
BENIGN                    4079
DoS Hulk                   338
PortScan                   304
DDoS                       220
FTP-Patator                 13
DoS Slowhttptest            10
DoS GoldenEye               10
SSH-Patator                  9
DoS slowloris                8
Bot                          4
Web Attack Brute Force       3
Web Attack XSS               2
Name: count, dtype: int64

BINARY PREDICTION COUNTS
binary_prediction
BENIGN    4078
ATTACK     922
Name: count, dtype: int64

HYBRID PREDICTION DISTRIBUTION
hybrid_prediction
BENIGN                    4078
DoS Hulk                   338
PortScan                   306
DDoS                       220
FTP-Patator                 13
DoS Slowhttptest            10
DoS GoldenEye               10
SSH-Patator                  9
DoS slowloris                8
Web Attack XSS     

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(


,true_label,binary_probability,binary_prediction,hybrid_prediction
3406,DDoS,9.999784e-01,ATTACK,DDoS
757,BENIGN,9.536234e-07,BENIGN,BENIGN
3624,BENIGN,5.898111e-07,BENIGN,BENIGN
4544,BENIGN,3.404217e-06,BENIGN,BENIGN
3235,PortScan,9.999685e-01,ATTACK,PortScan
1868,BENIGN,1.456775e-06,BENIGN,BENIGN
2916,BENIGN,1.301518e-06,BENIGN,BENIGN
3335,BENIGN,6.700826e-07,BENIGN,BENIGN
3527,BENIGN,5.440690e-04,BENIGN,BENIGN
2827,PortScan,9.982804e-01,ATTACK,PortScan


In [ ]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

BINARY_BUNDLE_PATH = (
    PROJECT_ROOT / "models/binary/xgboost_binary_bundle.joblib"
)
MULTICLASS_BUNDLE_PATH = (
    PROJECT_ROOT / "models/multiclass/xgboost_multiclass_bundle.joblib"
)
TEST_FILE = PROJECT_ROOT / "data/splits/test.parquet"
METRICS_DIR = PROJECT_ROOT / "outputs/metrics"
FIGURES_DIR = PROJECT_ROOT / "outputs/figures"

for path in [
    BINARY_BUNDLE_PATH,
    MULTICLASS_BUNDLE_PATH,
    TEST_FILE
]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load saved bundles
binary_bundle = joblib.load(BINARY_BUNDLE_PATH)
multiclass_bundle = joblib.load(MULTICLASS_BUNDLE_PATH)

binary_model = binary_bundle["model"]
binary_imputer = binary_bundle["imputer"]

multiclass_model = multiclass_bundle["model"]
multiclass_imputer = multiclass_bundle["imputer"]

binary_features = list(binary_bundle["feature_columns"])
multiclass_features = list(multiclass_bundle["feature_columns"])
class_names = list(multiclass_bundle["class_names"])

assert binary_features == multiclass_features, (
    "Binary and multiclass feature orders differ."
)
assert len(binary_features) == 78
assert len(class_names) == 15

# 2. Load held-out test features and labels
test_df = pd.read_parquet(TEST_FILE)

assert "label" in test_df.columns, "Test label column is missing."
assert all(c in test_df.columns for c in binary_features), (
    "Some required feature columns are missing."
)

X_test_df = test_df[binary_features].copy()
y_true = test_df["label"].astype(str).to_numpy()

unknown_labels = sorted(set(y_true) - set(class_names))
assert not unknown_labels, f"Unknown test labels: {unknown_labels}"

# Replace infinities before imputation
X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan)

# 3. Apply the saved preprocessing separately for each model
X_binary = binary_imputer.transform(X_test_df)
X_multiclass = multiclass_imputer.transform(X_test_df)

assert X_binary.shape == (len(test_df), 78)
assert X_multiclass.shape == (len(test_df), 78)
assert np.isfinite(X_binary).all()
assert np.isfinite(X_multiclass).all()

# 4. Binary detector: BENIGN vs ATTACK
binary_probabilities = binary_model.predict_proba(X_binary)

binary_classes = list(binary_bundle["label_mapping"].keys())
print("Binary label mapping:", binary_bundle["label_mapping"])

# The binary model's class-1 probability represents ATTACK
attack_probability = binary_probabilities[:, 1]
threshold = float(binary_bundle["decision_threshold"])

predicted_attack = attack_probability >= threshold

# 5. Multiclass model predicts the attack type
multiclass_probabilities = multiclass_model.predict_proba(X_multiclass)
multiclass_ids = np.argmax(multiclass_probabilities, axis=1)

# 6. Combine predictions into the hybrid output
y_pred = np.full(len(test_df), "BENIGN", dtype=object)

attack_indices = np.flatnonzero(predicted_attack)
y_pred[attack_indices] = np.asarray(class_names, dtype=object)[
    multiclass_ids[attack_indices]
]

# 7. Calculate full-test metrics
labels = class_names

results = {
    "test_records": int(len(y_true)),
    "binary_attack_threshold": threshold,
    "hybrid_accuracy": float(accuracy_score(y_true, y_pred)),
    "hybrid_balanced_accuracy": float(
        balanced_accuracy_score(y_true, y_pred)
    ),
    "hybrid_macro_f1": float(
        f1_score(
            y_true, y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        )
    ),
    "hybrid_weighted_f1": float(
        f1_score(
            y_true, y_pred,
            labels=labels,
            average="weighted",
            zero_division=0
        )
    ),
    "binary_predicted_attack_records": int(predicted_attack.sum()),
    "binary_predicted_benign_records": int((~predicted_attack).sum()),
    "class_names": labels,
    "confusion_matrix": confusion_matrix(
        y_true, y_pred, labels=labels
    ).tolist(),
    "classification_report": classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=labels,
        output_dict=True,
        zero_division=0
    )
}

print("\nFULL TEST-SET HYBRID EVALUATION")
print("=" * 60)
print(json.dumps(
    {k: v for k, v in results.items()
     if k not in ["confusion_matrix", "classification_report",
                  "class_names"]},
    indent=2
))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true, y_pred,
    labels=labels,
    target_names=labels,
    zero_division=0
))

# 8. Save results
output_path = METRICS_DIR / "hybrid_test_metrics.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("\nMetrics saved to:", output_path)


/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:58:50] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:58:50] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:58:50] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:58:52] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:58:52] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: U

Binary label mapping: {'BENIGN': 0, 'ATTACK': 1}

FULL TEST-SET HYBRID EVALUATION
{
  "test_records": 555056,
  "binary_attack_threshold": 0.5,
  "hybrid_accuracy": 0.9988974085497679,
  "hybrid_balanced_accuracy": 0.8124066051007447,
  "hybrid_macro_f1": 0.831223757372733,
  "hybrid_weighted_f1": 0.9988558263762252,
  "binary_predicted_attack_records": 103392,
  "binary_predicted_benign_records": 451664
}

CLASSIFICATION REPORT
                          precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00    451830
                     Bot       0.97      0.71      0.82       371
                    DDoS       1.00      1.00      1.00     25538
           DoS GoldenEye       1.00      0.99      1.00      2058
                DoS Hulk       1.00      1.00      1.00     38087
        DoS Slowhttptest       0.99      0.99      0.99      1058
           DoS slowloris       1.00      0.99      1.00      1106
             FTP-Patator       1.00   

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
METRICS_FILE = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "hybrid_test_metrics.json"
)

assert METRICS_FILE.exists(), (
    f"Metrics file not found: {METRICS_FILE}"
)

with open(METRICS_FILE, "r", encoding="utf-8") as f:
    hybrid_metrics = json.load(f)

print("HYBRID TEST METRICS")
print("=" * 50)

for key, value in hybrid_metrics.items():
    print(f"{key}: {value}")

print("\nMetrics file verification: PASSED")

HYBRID TEST METRICS
test_records: 555056
binary_attack_threshold: 0.5
hybrid_accuracy: 0.9988974085497679
hybrid_balanced_accuracy: 0.8124066051007447
hybrid_macro_f1: 0.831223757372733
hybrid_weighted_f1: 0.9988558263762252
binary_predicted_attack_records: 103392
binary_predicted_benign_records: 451664
class_names: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack Brute Force', 'Web Attack SQL Injection', 'Web Attack XSS']
confusion_matrix: [[451533, 8, 2, 4, 60, 7, 1, 0, 0, 0, 213, 0, 2, 0, 0], [108, 263, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 0, 25534, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 2046, 7, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [10, 0, 1, 2, 38068, 0, 0, 0, 0, 0, 5, 0, 1, 0, 0], [5, 0, 0, 0, 0, 1050, 2, 0, 0, 0, 0, 0, 1, 0, 0], [3, 0, 0, 0, 1, 2, 1099, 0, 0, 0, 0, 0, 1, 0, 0], [2, 0, 0, 0, 0, 0, 0, 1315, 0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0,

In [ ]:
required_variables = [
    "binary_bundle",
    "multiclass_bundle",
    "binary_model",
    "multiclass_model",
    "binary_imputer",
    "multiclass_imputer",
    "binary_features",
    "id_to_class",
    "TEST_FILE",
]

for name in required_variables:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

binary_bundle: AVAILABLE
multiclass_bundle: AVAILABLE
binary_model: AVAILABLE
multiclass_model: AVAILABLE
binary_imputer: AVAILABLE
multiclass_imputer: AVAILABLE
binary_features: AVAILABLE
id_to_class: MISSING
TEST_FILE: AVAILABLE


In [ ]:
# Restore the multiclass label mapping
id_to_class = {
    int(class_id): class_name
    for class_id, class_name
    in multiclass_bundle["id_to_class"].items()
}

print("Class mapping loaded:", len(id_to_class), "classes")

for class_id, class_name in id_to_class.items():
    print(f"{class_id} -> {class_name}")

assert len(id_to_class) == 15
assert id_to_class[0] == "BENIGN"

print("\nLabel mapping check: PASSED")

Class mapping loaded: 15 classes
0 -> BENIGN
1 -> Bot
2 -> DDoS
3 -> DoS GoldenEye
4 -> DoS Hulk
5 -> DoS Slowhttptest
6 -> DoS slowloris
7 -> FTP-Patator
8 -> Heartbleed
9 -> Infiltration
10 -> PortScan
11 -> SSH-Patator
12 -> Web Attack Brute Force
13 -> Web Attack SQL Injection
14 -> Web Attack XSS

Label mapping check: PASSED


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_true = results["true_label"]
y_pred = results["hybrid_prediction"]

print("SAMPLE HYBRID EVALUATION")
print("=" * 55)
print(f"Accuracy:          {accuracy_score(y_true, y_pred):.6f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.6f}")
print(f"Macro F1:           {f1_score(y_true, y_pred, average='macro', zero_division=0):.6f}")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,
    labels=list(id_to_class.values()),
    zero_division=0
))

print("\nCONFUSION MATRIX")
display(pd.DataFrame(
    confusion_matrix(
        y_true,
        y_pred,
        labels=list(id_to_class.values())
    ),
    index=list(id_to_class.values()),
    columns=list(id_to_class.values())
))

SAMPLE HYBRID EVALUATION
Accuracy:          0.999200
Balanced accuracy: 0.951348
Macro F1:           0.954458

CLASSIFICATION REPORT
                          precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00      4079
                     Bot       1.00      0.75      0.86         4
                    DDoS       1.00      1.00      1.00       220
           DoS GoldenEye       1.00      1.00      1.00        10
                DoS Hulk       1.00      1.00      1.00       338
        DoS Slowhttptest       1.00      1.00      1.00        10
           DoS slowloris       1.00      1.00      1.00         8
             FTP-Patator       1.00      1.00      1.00        13
              Heartbleed       0.00      0.00      0.00         0
            Infiltration       0.00      0.00      0.00         0
                PortScan       0.99      1.00      1.00       304
             SSH-Patator       1.00      1.00      1.00         9
  Web At

,BENIGN,Bot,DDoS,DoS GoldenEye,DoS Hulk,DoS Slowhttptest,DoS slowloris,FTP-Patator,Heartbleed,Infiltration,PortScan,SSH-Patator,Web Attack Brute Force,Web Attack SQL Injection,Web Attack XSS
BENIGN,4077,0,0,0,0,0,0,0,0,0,2,0,0,0,0
Bot,1,3,0,0,0,0,0,0,0,0,0,0,0,0,0
DDoS,0,0,220,0,0,0,0,0,0,0,0,0,0,0,0
DoS GoldenEye,0,0,0,10,0,0,0,0,0,0,0,0,0,0,0
DoS Hulk,0,0,0,0,338,0,0,0,0,0,0,0,0,0,0
DoS Slowhttptest,0,0,0,0,0,10,0,0,0,0,0,0,0,0,0
DoS slowloris,0,0,0,0,0,0,8,0,0,0,0,0,0,0,0
FTP-Patator,0,0,0,0,0,0,0,13,0,0,0,0,0,0,0
Heartbleed,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Infiltration,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

PROJECT_ROOT = "/content/drive/MyDrive/H-IDS"
TEST_FILE = f"{PROJECT_ROOT}/data/splits/test.parquet"

# Read true labels and the 78 model features
test_df = pd.read_parquet(TEST_FILE)

binary_features = list(binary_bundle["feature_columns"])
X_test = test_df[binary_features].replace(
    [np.inf, -np.inf], np.nan
)

y_true = test_df["label"].astype(str).to_numpy()
y_binary_true = (y_true != "BENIGN").astype(np.int8)

# Apply the saved binary imputer and detector
X_binary_test = binary_imputer.transform(X_test)
binary_probabilities = binary_model.predict_proba(
    X_binary_test
)[:, 1]

binary_threshold = float(
    binary_bundle["decision_threshold"]
)

y_binary_pred = (
    binary_probabilities >= binary_threshold
).astype(np.int8)

print("BINARY ROUTING AUDIT")
print("=" * 55)
print("Test records:", len(y_true))
print("Actual attacks:", int(y_binary_true.sum()))
print("Predicted attacks:", int(y_binary_pred.sum()))
print("Actual benign:", int((y_binary_true == 0).sum()))
print("Predicted benign:", int((y_binary_pred == 0).sum()))

print("\nConfusion matrix [BENIGN, ATTACK]:")
print(confusion_matrix(
    y_binary_true, y_binary_pred, labels=[0, 1]
))

print("\nBinary classification report:")
print(classification_report(
    y_binary_true,
    y_binary_pred,
    labels=[0, 1],
    target_names=["BENIGN", "ATTACK"],
    digits=5,
    zero_division=0
))

# Identify attacks that the binary detector misses
missed_attacks = test_df.loc[
    (y_binary_true == 1) & (y_binary_pred == 0),
    ["label"]
].copy()

print("\nATTACKS MISSED BY BINARY DETECTOR")
print(missed_attacks["label"].value_counts())

del X_test, X_binary_test, test_df

BINARY ROUTING AUDIT
Test records: 555056
Actual attacks: 103226
Predicted attacks: 103392
Actual benign: 451830
Predicted benign: 451664

Confusion matrix [BENIGN, ATTACK]:
[[451522    308]
 [   142 103084]]

Binary classification report:
              precision    recall  f1-score   support

      BENIGN    0.99969   0.99932   0.99950    451830
      ATTACK    0.99702   0.99862   0.99782    103226

    accuracy                        0.99919    555056
   macro avg    0.99835   0.99897   0.99866    555056
weighted avg    0.99919   0.99919   0.99919    555056


ATTACKS MISSED BY BINARY DETECTOR
label
Bot                         108
DoS Hulk                      7
Infiltration                  6
DDoS                          4
DoS Slowhttptest              4
Web Attack Brute Force        4
DoS slowloris                 3
Web Attack XSS                2
PortScan                      1
Web Attack SQL Injection      1
FTP-Patator                   1
DoS GoldenEye                 1
Name: co

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
TEST_FILE = PROJECT_ROOT / "data/splits/test.parquet"

# Load the test labels and feature columns required by the binary model
binary_features = list(binary_bundle["feature_columns"])

test_df = pd.read_parquet(
    TEST_FILE,
    columns=binary_features + ["label"]
)

y_true = test_df["label"].astype(str).to_numpy()
actual_attack = y_true != "BENIGN"

X_test = test_df[binary_features].replace(
    [np.inf, -np.inf], np.nan
)

X_test_imputed = binary_imputer.transform(X_test)

# Calculate attack probabilities once
attack_prob = binary_model.predict_proba(
    X_test_imputed
)[:, 1]

threshold = float(binary_bundle["decision_threshold"])
predicted_attack = attack_prob >= threshold

# Identify attacks missed at the saved threshold
miss_mask = actual_attack & ~predicted_attack

missed_df = pd.DataFrame({
    "true_label": y_true[miss_mask],
    "attack_probability": attack_prob[miss_mask]
})

print("MISSED ATTACK PROBABILITY SUMMARY")
print("=" * 55)
print("Missed attacks:", len(missed_df))

print("\nProbability statistics:")
print(missed_df["attack_probability"].describe())

print("\nMissed attacks by class and probability:")
print(
    missed_df.groupby("true_label")["attack_probability"]
    .agg(["count", "min", "median", "max"])
    .sort_values("count", ascending=False)
)

print("\nTHRESHOLD COMPARISON")
print("=" * 55)

for t in [0.50, 0.40, 0.30, 0.20, 0.10]:
    pred = attack_prob >= t

    tp = int(np.sum(actual_attack & pred))
    fn = int(np.sum(actual_attack & ~pred))
    fp = int(np.sum(~actual_attack & pred))
    tn = int(np.sum(~actual_attack & ~pred))

    recall = tp / (tp + fn) if tp + fn else 0
    precision = tp / (tp + fp) if tp + fp else 0

    print(
        f"Threshold={t:.2f} | "
        f"Attack recall={recall:.5%} | "
        f"Attack precision={precision:.5%} | "
        f"Missed attacks={fn:,} | "
        f"False alarms={fp:,}"
    )

del X_test, X_test_imputed, test_df

MISSED ATTACK PROBABILITY SUMMARY
Missed attacks: 142

Probability statistics:
count    142.000000
mean       0.319526
std        0.143464
min        0.000182
25%        0.259606
50%        0.371143
75%        0.424304
max        0.499390
Name: attack_probability, dtype: float64

Missed attacks by class and probability:
                          count       min    median       max
true_label                                                   
Bot                         108  0.001203  0.376614  0.499390
DoS Hulk                      7  0.000182  0.458905  0.497658
Infiltration                  6  0.004462  0.018574  0.143592
DDoS                          4  0.002434  0.125319  0.284494
Web Attack Brute Force        4  0.053292  0.264439  0.421455
DoS Slowhttptest              4  0.014941  0.209221  0.294906
DoS slowloris                 3  0.004587  0.050429  0.459415
Web Attack XSS                2  0.152614  0.181140  0.209667
DoS GoldenEye                 1  0.456138  0.456138  0.456

In [5]:

from pathlib import Path
import joblib

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

BINARY_BUNDLE_PATH = (
    PROJECT_ROOT / "models/binary/xgboost_binary_bundle.joblib"
)
MULTICLASS_BUNDLE_PATH = (
    PROJECT_ROOT / "models/multiclass/xgboost_multiclass_bundle.joblib"
)

# Check that Google Drive is mounted and the files exist
assert Path("/content/drive/MyDrive").exists(), (
    "Google Drive is not mounted. Mount it before continuing."
)
assert BINARY_BUNDLE_PATH.exists(), (
    f"Binary bundle not found: {BINARY_BUNDLE_PATH}"
)
assert MULTICLASS_BUNDLE_PATH.exists(), (
    f"Multiclass bundle not found: {MULTICLASS_BUNDLE_PATH}"
)

# Reload the saved bundles
binary_bundle = joblib.load(BINARY_BUNDLE_PATH)
multiclass_bundle = joblib.load(MULTICLASS_BUNDLE_PATH)

# Restore the variables needed for evaluation
binary_model = binary_bundle["model"]
binary_imputer = binary_bundle["imputer"]
binary_features = list(binary_bundle["feature_columns"])
binary_threshold = float(binary_bundle["decision_threshold"])

multiclass_model = multiclass_bundle["model"]
multiclass_imputer = multiclass_bundle["imputer"]
multiclass_features = list(multiclass_bundle["feature_columns"])

print("Binary bundle loaded:", True)
print("Multiclass bundle loaded:", True)
print("Binary features:", len(binary_features))
print("Multiclass features:", len(multiclass_features))
print("Saved binary threshold:", binary_threshold)

assert binary_features == multiclass_features
assert len(binary_features) == 78

print("Bundle and feature-order checks: PASSED")


Binary bundle loaded: True
Multiclass bundle loaded: True
Binary features: 78
Multiclass features: 78
Saved binary threshold: 0.5
Bundle and feature-order checks: PASSED


/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:17:33] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:17:33] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:17:33] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:17:34] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [05:17:34] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: U

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_score, recall_score

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
VALIDATION_FILE = PROJECT_ROOT / "data/splits/validation.parquet"

binary_features = list(binary_bundle["feature_columns"])

val_df = pd.read_parquet(
    VALIDATION_FILE,
    columns=binary_features + ["label"]
)

y_val_labels = val_df["label"].astype(str).to_numpy()
actual_attack = y_val_labels != "BENIGN"

X_val = val_df[binary_features].replace(
    [np.inf, -np.inf], np.nan
)
X_val_imputed = binary_imputer.transform(X_val)

attack_prob = binary_model.predict_proba(
    X_val_imputed
)[:, 1]

thresholds = [0.50, 0.40, 0.30, 0.20, 0.10]
rows = []

for threshold in thresholds:
    predicted_attack = attack_prob >= threshold

    missed = actual_attack & ~predicted_attack
    false_alarms = ~actual_attack & predicted_attack

    rows.append({
        "threshold": threshold,
        "attack_recall": recall_score(
            actual_attack, predicted_attack, zero_division=0
        ),
        "attack_precision": precision_score(
            actual_attack, predicted_attack, zero_division=0
        ),
        "missed_attacks": int(missed.sum()),
        "false_alarms": int(false_alarms.sum()),
        "predicted_attacks": int(predicted_attack.sum())
    })

    if threshold == 0.30:
        missed_labels_at_030 = pd.Series(
            y_val_labels[missed]
        ).value_counts()

print("VALIDATION THRESHOLD COMPARISON")
print("=" * 70)

comparison = pd.DataFrame(rows)
display(
    comparison.style.format({
        "threshold": "{:.2f}",
        "attack_recall": "{:.5%}",
        "attack_precision": "{:.5%}"
    })
)

print("\nATTACK CLASSES MISSED AT THRESHOLD 0.30")
print("=" * 55)
print(missed_labels_at_030)

del val_df, X_val, X_val_imputed

VALIDATION THRESHOLD COMPARISON


,threshold,attack_recall,attack_precision,missed_attacks,false_alarms,predicted_attacks
0,0.50,99.88486%,99.75502%,69,147,60005
1,0.40,99.93492%,99.68872%,39,187,60075
2,0.30,99.97330%,99.61591%,16,231,60142
3,0.20,99.98999%,99.58286%,6,251,60172
4,0.10,99.99499%,99.54153%,3,276,60200



ATTACK CLASSES MISSED AT THRESHOLD 0.30
Bot                 13
Infiltration         1
DoS slowloris        1
DoS Slowhttptest     1
Name: count, dtype: int64


In [ ]:
# Inspect attack probabilities for Bot records missed at threshold 0.30
threshold_candidate = 0.30

missed_mask = (
    (y_val_labels != "BENIGN")
    & (attack_prob < threshold_candidate)
)

missed_validation = pd.DataFrame({
    "true_label": y_val_labels[missed_mask],
    "attack_probability": attack_prob[missed_mask]
})

print("Missed validation attacks at threshold 0.30")
print("=" * 55)
print(missed_validation["true_label"].value_counts())

print("\nBot probability details")
bot_missed = missed_validation[
    missed_validation["true_label"] == "Bot"
]

print("Bot records missed:", len(bot_missed))
if not bot_missed.empty:
    display(
        bot_missed.sort_values("attack_probability")
    )

Missed validation attacks at threshold 0.30
true_label
Bot                 13
Infiltration         1
DoS slowloris        1
DoS Slowhttptest     1
Name: count, dtype: int64

Bot probability details
Bot records missed: 13


,true_label,attack_probability
5,Bot,0.000436
0,Bot,0.117436
3,Bot,0.119169
9,Bot,0.139207
6,Bot,0.215060
11,Bot,0.232852
7,Bot,0.239135
2,Bot,0.240155
1,Bot,0.241585
4,Bot,0.245682


In [7]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
VALIDATION_FILE = PROJECT_ROOT / "data/splits/validation.parquet"

assert VALIDATION_FILE.exists(), (
    f"Validation file not found: {VALIDATION_FILE}"
)

# Restore validation labels and binary features
binary_features = list(binary_bundle["feature_columns"])

val_df = pd.read_parquet(
    VALIDATION_FILE,
    columns=binary_features + ["label"]
)

y_val_labels = val_df["label"].astype(str).to_numpy()

# Recompute probabilities using the saved model and imputer
X_val = val_df[binary_features].replace(
    [np.inf, -np.inf], np.nan
)

X_val_imputed = binary_imputer.transform(X_val)
attack_prob = binary_model.predict_proba(
    X_val_imputed
)[:, 1]

assert len(y_val_labels) == len(attack_prob)
assert np.isfinite(attack_prob).all()

print("Validation data reloaded:", len(y_val_labels))
print("Bot records:", int(np.sum(y_val_labels == "Bot")))
print("Validation probability checks: PASSED")


Validation data reloaded: 285201
Bot records: 188
Validation probability checks: PASSED


In [8]:
import pandas as pd
import numpy as np

bot_mask = y_val_labels == "Bot"
bot_probabilities = attack_prob[bot_mask]

print("BOT THRESHOLD ANALYSIS")
print("=" * 55)
print("Total Bot records in validation:", int(bot_mask.sum()))

for threshold in [0.30, 0.25, 0.20, 0.15, 0.10, 0.05, 0.01]:
    detected = bot_probabilities >= threshold
    missed = int((~detected).sum())
    detected_count = int(detected.sum())
    recall = detected_count / len(bot_probabilities)

    print(
        f"Threshold={threshold:.2f} | "
        f"Detected={detected_count} | "
        f"Missed={missed} | "
        f"Bot recall={recall:.2%}"
    )

BOT THRESHOLD ANALYSIS
Total Bot records in validation: 188
Threshold=0.30 | Detected=175 | Missed=13 | Bot recall=93.09%
Threshold=0.25 | Detected=178 | Missed=10 | Bot recall=94.68%
Threshold=0.20 | Detected=184 | Missed=4 | Bot recall=97.87%
Threshold=0.15 | Detected=184 | Missed=4 | Bot recall=97.87%
Threshold=0.10 | Detected=187 | Missed=1 | Bot recall=99.47%
Threshold=0.05 | Detected=187 | Missed=1 | Bot recall=99.47%
Threshold=0.01 | Detected=187 | Missed=1 | Bot recall=99.47%


In [9]:

from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"

split_files = {
    "train": SPLIT_DIR / "train.parquet",
    "validation": SPLIT_DIR / "validation.parquet",
    "test": SPLIT_DIR / "test.parquet",
}

for name, path in split_files.items():
    assert path.exists(), f"Missing {name} split: {path}"

feature_columns = list(binary_bundle["feature_columns"])

# Build a compact set of feature hashes for each split.
# Labels are kept separately to identify conflicting-label overlaps.
hashes_by_split = {}
labels_by_split = {}

for split_name, path in split_files.items():
    print(f"Hashing {split_name} split...")

    hash_chunks = []
    label_chunks = []

    for chunk in pd.read_parquet(
        path,
        columns=feature_columns + ["label"]
    ).pipe(lambda df: [df]):
        feature_df = chunk[feature_columns].replace(
            [np.inf, -np.inf], np.nan
        )

        # Stable row hashes; identical feature rows receive identical hashes.
        row_hashes = pd.util.hash_pandas_object(
            feature_df,
            index=False
        ).to_numpy(dtype=np.uint64)

        hash_chunks.append(row_hashes)
        label_chunks.append(
            chunk["label"].astype(str).to_numpy()
        )

    hashes_by_split[split_name] = np.concatenate(hash_chunks)
    labels_by_split[split_name] = np.concatenate(label_chunks)

    print(
        f"{split_name}: {len(hashes_by_split[split_name]):,} rows, "
        f"{len(np.unique(hashes_by_split[split_name])):,} unique hashes"
    )

print("\nCROSS-SPLIT HASH OVERLAP")
print("=" * 55)

for left, right in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    left_hashes = set(hashes_by_split[left].tolist())
    right_hashes = set(hashes_by_split[right].tolist())

    overlap = left_hashes & right_hashes

    print(
        f"{left} vs {right}: "
        f"{len(overlap):,} shared unique feature hashes"
    )

print("\nHash audit completed. No files were modified.")


Hashing train split...
train: 1,990,486 rows, 1,765,164 unique hashes
Hashing validation split...
validation: 285,201 rows, 252,167 unique hashes
Hashing test split...
test: 555,056 rows, 504,333 unique hashes

CROSS-SPLIT HASH OVERLAP
train vs validation: 0 shared unique feature hashes
train vs test: 0 shared unique feature hashes
validation vs test: 0 shared unique feature hashes

Hash audit completed. No files were modified.


In [10]:

variables_to_check = [
    "binary_bundle",
    "multiclass_bundle",
    "binary_model",
    "multiclass_model",
    "binary_imputer",
    "multiclass_imputer",
    "binary_features",
    "multiclass_features",
    "y_val_labels",
    "attack_prob",
    "X_val",
    "X_val_imputed",
    "y_val",
    "y_pred_multi_val",
    "multiclass_val_predictions",
    "hybrid_val_predictions",
]

print("NOTEBOOK VARIABLE CHECK")
print("=" * 55)

for name in variables_to_check:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'MISSING'}")


NOTEBOOK VARIABLE CHECK
binary_bundle: AVAILABLE
multiclass_bundle: AVAILABLE
binary_model: AVAILABLE
multiclass_model: AVAILABLE
binary_imputer: AVAILABLE
multiclass_imputer: AVAILABLE
binary_features: AVAILABLE
multiclass_features: AVAILABLE
y_val_labels: AVAILABLE
attack_prob: AVAILABLE
X_val: AVAILABLE
X_val_imputed: AVAILABLE
y_val: MISSING
y_pred_multi_val: MISSING
multiclass_val_predictions: MISSING
hybrid_val_predictions: MISSING


In [11]:

import numpy as np

print("VALIDATION ARRAY CHECK")
print("=" * 55)

for name in [
    "y_val_labels",
    "attack_prob",
    "y_pred_multi_val",
    "multiclass_val_predictions",
    "hybrid_val_predictions",
]:
    value = globals().get(name)

    if value is None:
        print(f"{name}: unavailable")
    else:
        print(f"{name}: shape={np.shape(value)}")


VALIDATION ARRAY CHECK
y_val_labels: shape=(285201,)
attack_prob: shape=(285201,)
y_pred_multi_val: unavailable
multiclass_val_predictions: unavailable
hybrid_val_predictions: unavailable


In [12]:

import numpy as np
import pandas as pd

# Verify the existing validation data
assert len(y_val_labels) == len(attack_prob) == 285201
assert len(multiclass_features) == 78

# Prepare the validation features in the multiclass model's expected order
X_val_multi = X_val[multiclass_features].replace(
    [np.inf, -np.inf], np.nan
)

X_val_multi_imputed = multiclass_imputer.transform(X_val_multi)

# Predict multiclass IDs for all validation records.
# We will use these predictions only for records routed as attacks.
multiclass_val_predictions = multiclass_model.predict(
    X_val_multi_imputed
).astype(np.int64)

assert len(multiclass_val_predictions) == len(y_val_labels)
assert np.isfinite(multiclass_val_predictions).all()

print("Multiclass validation predictions generated.")
print("Validation records:", len(y_val_labels))
print("Prediction shape:", multiclass_val_predictions.shape)
print("Predicted class IDs:", np.unique(multiclass_val_predictions))


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(


Multiclass validation predictions generated.
Validation records: 285201
Prediction shape: (285201,)
Predicted class IDs: [ 0  1  2  3  4  5  6  7  9 10 11 12 13 14]


In [13]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder

# Convert the saved multiclass class IDs back to label strings
id_to_class = multiclass_bundle["id_to_class"]

if all(str(k).isdigit() for k in id_to_class):
    class_names = {
        int(k): v for k, v in id_to_class.items()
    }
else:
    class_names = {
        int(v): k for k, v in multiclass_bundle["class_to_id"].items()
    }

multi_labels = np.array([
    class_names[int(class_id)]
    for class_id in multiclass_val_predictions
])

# Binary routing thresholds to compare
thresholds = [0.50, 0.30, 0.20, 0.10]
rows = []

for threshold in thresholds:
    binary_attack = attack_prob >= threshold

    # Default every record to BENIGN, then route detected attacks
    hybrid_labels = np.full(
        len(y_val_labels), "BENIGN", dtype=object
    )
    hybrid_labels[binary_attack] = multi_labels[binary_attack]

    actual_attack = y_val_labels != "BENIGN"
    predicted_attack = hybrid_labels != "BENIGN"

    cm_binary = confusion_matrix(
        actual_attack.astype(int),
        binary_attack.astype(int),
        labels=[0, 1]
    )

    # End-to-end metrics across the 15-class label space
    rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_val_labels, hybrid_labels),
        "balanced_accuracy": balanced_accuracy_score(
            y_val_labels, hybrid_labels
        ),
        "macro_f1": f1_score(
            y_val_labels,
            hybrid_labels,
            labels=list(class_names.values()),
            average="macro",
            zero_division=0
        ),
        "attack_recall": recall_score(
            actual_attack, predicted_attack, zero_division=0
        ),
        "missed_attacks": int(
            np.sum(actual_attack & ~predicted_attack)
        ),
        "false_alarms": int(
            np.sum(~actual_attack & predicted_attack)
        ),
        "binary_false_positives": int(cm_binary[0, 1]),
        "predicted_attacks": int(binary_attack.sum())
    })

threshold_results = pd.DataFrame(rows)

print("VALIDATION HYBRID THRESHOLD COMPARISON")
display(
    threshold_results.style.format({
        "threshold": "{:.2f}",
        "accuracy": "{:.6f}",
        "balanced_accuracy": "{:.6f}",
        "macro_f1": "{:.6f}",
        "attack_recall": "{:.6%}",
    })
)


VALIDATION HYBRID THRESHOLD COMPARISON


,threshold,accuracy,balanced_accuracy,macro_f1,attack_recall,missed_attacks,false_alarms,binary_false_positives,predicted_attacks
0,0.50,0.998983,0.843053,0.814913,99.871510%,77,141,147,60005
1,0.30,0.998966,0.846940,0.814403,99.894872%,63,160,231,60142
2,0.20,0.998966,0.846940,0.814403,99.894872%,63,160,251,60172
3,0.10,0.998962,0.846939,0.814343,99.894872%,63,161,276,60200


In [14]:

from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

def make_hybrid_predictions(threshold):
    attack_mask = attack_prob >= threshold

    predictions = np.full(
        len(y_val_labels), "BENIGN", dtype=object
    )
    predictions[attack_mask] = multi_labels[attack_mask]

    return predictions

pred_050 = make_hybrid_predictions(0.50)
pred_030 = make_hybrid_predictions(0.30)

report_050 = classification_report(
    y_val_labels,
    pred_050,
    labels=list(class_names.values()),
    output_dict=True,
    zero_division=0
)

report_030 = classification_report(
    y_val_labels,
    pred_030,
    labels=list(class_names.values()),
    output_dict=True,
    zero_division=0
)

comparison = pd.DataFrame({
    "recall_at_0.50": {
        name: report_050[name]["recall"]
        for name in class_names.values()
    },
    "recall_at_0.30": {
        name: report_030[name]["recall"]
        for name in class_names.values()
    },
})

comparison["change"] = (
    comparison["recall_at_0.30"]
    - comparison["recall_at_0.50"]
)

print("PER-CLASS RECALL COMPARISON")
display(comparison.sort_values("change"))

print("\nBOT AND RARE ATTACK CLASSES")
display(
    comparison.loc[
        comparison.index.isin([
            "Bot",
            "Heartbleed",
            "Infiltration",
            "Web Attack SQL Injection",
            "Web Attack XSS",
        ])
    ]
)


PER-CLASS RECALL COMPARISON


,recall_at_0.50,recall_at_0.30,change
BENIGN,0.999374,0.999290,-0.000084
DDoS,1.000000,1.000000,0.000000
DoS Slowhttptest,0.990893,0.990893,0.000000
DoS Hulk,0.999818,0.999818,0.000000
DoS slowloris,0.992780,0.992780,0.000000
PortScan,0.999744,0.999744,0.000000
Infiltration,0.666667,0.666667,0.000000
Heartbleed,0.000000,0.000000,0.000000
Web Attack XSS,0.313725,0.313725,0.000000
Web Attack SQL Injection,0.333333,0.333333,0.000000



BOT AND RARE ATTACK CLASSES


,recall_at_0.50,recall_at_0.30,change
Bot,0.696809,0.744681,0.047872
Heartbleed,0.000000,0.000000,0.000000
Infiltration,0.666667,0.666667,0.000000
Web Attack SQL Injection,0.333333,0.333333,0.000000
Web Attack XSS,0.313725,0.313725,0.000000


In [15]:

import numpy as np
import pandas as pd

threshold = 0.30
bot_mask = y_val_labels == "Bot"
binary_attack = attack_prob >= threshold

bot_detected_by_binary = bot_mask & binary_attack
bot_missed_by_binary = bot_mask & ~binary_attack

bot_correct_multiclass = (
    bot_detected_by_binary
    & (multi_labels == "Bot")
)

bot_wrong_multiclass = (
    bot_detected_by_binary
    & (multi_labels != "Bot")
)

print("BOT ERROR BREAKDOWN AT THRESHOLD 0.30")
print("=" * 55)
print("Actual Bot records:", int(bot_mask.sum()))
print("Missed by binary detector:", int(bot_missed_by_binary.sum()))
print("Detected by binary detector:", int(bot_detected_by_binary.sum()))
print("Correctly classified as Bot:", int(bot_correct_multiclass.sum()))
print("Misclassified by multiclass model:", int(bot_wrong_multiclass.sum()))

print("\nMulticlass predictions for detected Bot records:")
print(
    pd.Series(multi_labels[bot_detected_by_binary])
    .value_counts()
)


BOT ERROR BREAKDOWN AT THRESHOLD 0.30
Actual Bot records: 188
Missed by binary detector: 13
Detected by binary detector: 175
Correctly classified as Bot: 140
Misclassified by multiclass model: 35

Multiclass predictions for detected Bot records:
Bot       140
BENIGN     35
Name: count, dtype: int64


In [16]:

import numpy as np
import pandas as pd

threshold = 0.30

bot_mask = y_val_labels == "Bot"
binary_attack = attack_prob >= threshold

# Bot records routed to multiclass but classified as BENIGN
error_mask = (
    bot_mask
    & binary_attack
    & (multi_labels == "BENIGN")
)

error_indices = np.flatnonzero(error_mask)

print("Bot records misclassified as BENIGN")
print("=" * 55)
print("Number of errors:", len(error_indices))

# Get class probabilities for these records
if len(error_indices):
    probabilities = multiclass_model.predict_proba(
        X_val_multi_imputed[error_indices]
    )

    benign_class_id = multiclass_bundle["class_to_id"]["BENIGN"]
    bot_class_id = multiclass_bundle["class_to_id"]["Bot"]

    error_details = pd.DataFrame({
        "validation_position": error_indices,
        "binary_attack_probability": attack_prob[error_indices],
        "multiclass_benign_probability": probabilities[:, benign_class_id],
        "multiclass_bot_probability": probabilities[:, bot_class_id],
    })

    error_details["bot_probability_rank"] = (
        probabilities.argsort(axis=1)[:, ::-1]
    ).tolist()

    print("\nProbability summary:")
    display(
        error_details[
            [
                "binary_attack_probability",
                "multiclass_benign_probability",
                "multiclass_bot_probability",
            ]
        ].describe()
    )

    print("\nFirst 15 errors:")
    display(error_details.head(15))


Bot records misclassified as BENIGN
Number of errors: 35

Probability summary:


,binary_attack_probability,multiclass_benign_probability,multiclass_bot_probability
count,35.000000,35.000000,35.000000
mean,0.388506,0.655483,0.343972
std,0.043200,0.087352,0.087421
min,0.309382,0.503673,0.143695
25%,0.351431,0.588900,0.291665
50%,0.388351,0.648727,0.350712
75%,0.434454,0.707743,0.410605
max,0.459292,0.855617,0.495897



First 15 errors:


,validation_position,binary_attack_probability,multiclass_benign_probability,multiclass_bot_probability,bot_probability_rank
0,63462,0.352691,0.679871,0.319553,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 6, 14,..."
1,63844,0.343774,0.719230,0.280168,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 6, 14,..."
2,64334,0.440541,0.648727,0.350712,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 14, 6,..."
3,64703,0.438711,0.561571,0.437963,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 6, 14,..."
4,64727,0.394827,0.679211,0.320204,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 14, 6,..."
5,64740,0.315090,0.692333,0.307105,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 6, 14,..."
6,65200,0.393995,0.535530,0.464036,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 6, 14,..."
7,65857,0.309382,0.798157,0.201206,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 14, 6,..."
8,66311,0.459292,0.577580,0.421918,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 14, 6,..."
9,66696,0.350172,0.798086,0.201259,"[0, 1, 10, 4, 9, 8, 2, 7, 11, 13, 3, 5, 14, 6,..."


In [17]:

# Analyze the 35 Bot records misclassified as BENIGN
error_indices = np.flatnonzero(
    (y_val_labels == "Bot")
    & (attack_prob >= 0.30)
    & (multi_labels == "BENIGN")
)

probabilities = multiclass_model.predict_proba(
    X_val_multi_imputed[error_indices]
)

class_to_id = multiclass_bundle["class_to_id"]
id_to_class = {
    int(v): k for k, v in class_to_id.items()
}

benign_id = class_to_id["BENIGN"]
bot_id = class_to_id["Bot"]

top_two = np.argsort(probabilities, axis=1)[:, -2:][:, ::-1]

diagnostics = pd.DataFrame({
    "validation_position": error_indices,
    "binary_attack_probability": attack_prob[error_indices],
    "benign_probability": probabilities[:, benign_id],
    "bot_probability": probabilities[:, bot_id],
    "benign_minus_bot": (
        probabilities[:, benign_id] - probabilities[:, bot_id]
    ),
    "top_predicted_class": [
        id_to_class[int(row[0])] for row in top_two
    ],
    "second_predicted_class": [
        id_to_class[int(row[1])] for row in top_two
    ],
    "top_probability": [
        probabilities[i, row[0]]
        for i, row in enumerate(top_two)
    ],
    "second_probability": [
        probabilities[i, row[1]]
        for i, row in enumerate(top_two)
    ],
})

print("Bot misclassification diagnostics")
print("Records:", len(diagnostics))
print("\nTop predicted classes:")
print(diagnostics["top_predicted_class"].value_counts())

print("\nConfidence summary:")
display(
    diagnostics[
        [
            "benign_probability",
            "bot_probability",
            "benign_minus_bot",
            "top_probability",
            "second_probability",
        ]
    ].describe()
)

print("\nRecords with the smallest BENIGN vs Bot probability gap:")
display(
    diagnostics.sort_values("benign_minus_bot").head(10)
)


Bot misclassification diagnostics
Records: 35

Top predicted classes:
top_predicted_class
BENIGN    35
Name: count, dtype: int64

Confidence summary:


,benign_probability,bot_probability,benign_minus_bot,top_probability,second_probability
count,35.000000,35.000000,35.000000,35.000000,35.000000
mean,0.655483,0.343972,0.311511,0.655483,0.343972
std,0.087352,0.087421,0.174773,0.087352,0.087421
min,0.503673,0.143695,0.007777,0.503673,0.143695
25%,0.588900,0.291665,0.178295,0.588900,0.291665
50%,0.648727,0.350712,0.298015,0.648727,0.350712
75%,0.707743,0.410605,0.416077,0.707743,0.410605
max,0.855617,0.495897,0.711922,0.855617,0.495897



Records with the smallest BENIGN vs Bot probability gap:


,validation_position,binary_attack_probability,benign_probability,bot_probability,benign_minus_bot,top_predicted_class,second_predicted_class,top_probability,second_probability
22,69559,0.436034,0.503673,0.495897,0.007777,BENIGN,Bot,0.503673,0.495897
30,69812,0.419787,0.533343,0.466211,0.067132,BENIGN,Bot,0.533343,0.466211
6,65200,0.393995,0.535530,0.464036,0.071494,BENIGN,Bot,0.535530,0.464036
32,69825,0.449075,0.538431,0.461101,0.077330,BENIGN,Bot,0.538431,0.461101
19,68841,0.367193,0.559422,0.440131,0.119291,BENIGN,Bot,0.559422,0.440131
3,64703,0.438711,0.561571,0.437963,0.123608,BENIGN,Bot,0.561571,0.437963
14,67375,0.441012,0.563708,0.435816,0.127892,BENIGN,Bot,0.563708,0.435816
8,66311,0.459292,0.577580,0.421918,0.155662,BENIGN,Bot,0.577580,0.421918
17,68230,0.414343,0.588779,0.410727,0.178052,BENIGN,Bot,0.588779,0.410727
10,66895,0.388351,0.589021,0.410484,0.178537,BENIGN,Bot,0.589021,0.410484


In [18]:

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# Existing validation predictions and probabilities
threshold_binary = 0.30
binary_attack = attack_prob >= threshold_binary

# Multiclass probabilities for all validation records
multi_probabilities = multiclass_model.predict_proba(
    X_val_multi_imputed
)

bot_id = multiclass_bundle["class_to_id"]["Bot"]
bot_probability = multi_probabilities[:, bot_id]

actual_bot = y_val_labels == "Bot"
results = []

# Apply the override only to records routed to multiclass
for bot_threshold in [0.30, 0.35, 0.40, 0.45, 0.49]:
    hybrid_predictions = np.full(
        len(y_val_labels), "BENIGN", dtype=object
    )
    hybrid_predictions[binary_attack] = multi_labels[binary_attack]

    # Experimental Bot override
    bot_override = (
        binary_attack
        & (bot_probability >= bot_threshold)
    )
    hybrid_predictions[bot_override] = "Bot"

    predicted_bot = hybrid_predictions == "Bot"

    results.append({
        "bot_threshold": bot_threshold,
        "bot_recall": recall_score(
            actual_bot, predicted_bot, zero_division=0
        ),
        "bot_precision": precision_score(
            actual_bot, predicted_bot, zero_division=0
        ),
        "actual_bot_records": int(actual_bot.sum()),
        "predicted_bot_records": int(predicted_bot.sum()),
        "non_bot_records_mislabeled_bot": int(
            np.sum(predicted_bot & ~actual_bot)
        ),
        "bot_records_missed": int(
            np.sum(actual_bot & ~predicted_bot)
        )
    })

print("EXPERIMENTAL BOT OVERRIDE")
print("Binary routing threshold:", threshold_binary)
display(pd.DataFrame(results).style.format({
    "bot_threshold": "{:.2f}",
    "bot_recall": "{:.2%}",
    "bot_precision": "{:.2%}"
}))


EXPERIMENTAL BOT OVERRIDE
Binary routing threshold: 0.3


,bot_threshold,bot_recall,bot_precision,actual_bot_records,predicted_bot_records,non_bot_records_mislabeled_bot,bot_records_missed
0,0.30,88.30%,73.13%,188,227,61,22
1,0.35,84.04%,75.60%,188,209,51,30
2,0.40,79.79%,77.32%,188,194,44,38
3,0.45,76.60%,83.72%,188,172,28,44
4,0.49,75.00%,85.98%,188,164,23,47


In [20]:

import numpy as np

BINARY_THRESHOLD = 0.30

# Binary attack probabilities are already available as attack_prob.
binary_attack = np.asarray(attack_prob) >= BINARY_THRESHOLD

# Generate multiclass predictions for the validation records.
# Use the same row order as y_val_labels.
multi_predictions = np.asarray(
    multiclass_model.predict(X_val_multi_imputed),
    dtype=np.int64
)

# Convert class IDs to class names.
class_names = list(multiclass_bundle["class_names"])
multi_labels = np.asarray(class_names, dtype=object)[multi_predictions]

# Reconstruct original end-to-end hybrid predictions.
hybrid_val_predictions = np.full(
    len(y_val_labels), "BENIGN", dtype=object
)

hybrid_val_predictions[binary_attack] = multi_labels[binary_attack]

# Validate array alignment.
assert len(hybrid_val_predictions) == len(y_val_labels)
assert len(attack_prob) == len(y_val_labels)
assert len(multi_labels) == len(y_val_labels)

print("Hybrid validation predictions reconstructed.")
print("Validation records:", len(y_val_labels))
print("Records routed to multiclass:", int(binary_attack.sum()))
print("Hybrid prediction shape:", hybrid_val_predictions.shape)
print("Prediction labels:", np.unique(hybrid_val_predictions))


Hybrid validation predictions reconstructed.
Validation records: 285201
Records routed to multiclass: 60142
Hybrid prediction shape: (285201,)
Prediction labels: ['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Infiltration' 'PortScan' 'SSH-Patator'
 'Web Attack Brute Force' 'Web Attack SQL Injection' 'Web Attack XSS']


In [21]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

bot_threshold = 0.40
binary_threshold = 0.30

actual = np.asarray(y_val_labels)
original = np.asarray(hybrid_val_predictions).copy()

# Use the same binary routing rule as the experiment
binary_attack = attack_prob >= binary_threshold
bot_id = multiclass_bundle["class_to_id"]["Bot"]

# Experimental override
bot_probability = multiclass_model.predict_proba(
    X_val_multi_imputed
)[:, bot_id]

adjusted = original.copy()
bot_override = (
    binary_attack
    & (bot_probability >= bot_threshold)
)
adjusted[bot_override] = "Bot"

for name, predictions in [
    ("Original hybrid", original),
    ("Bot override", adjusted),
]:
    print(f"\n{name}")
    print("=" * 55)
    print("Accuracy:", round(accuracy_score(actual, predictions), 6))
    print(
        "Balanced accuracy:",
        round(balanced_accuracy_score(actual, predictions), 6),
    )
    print(
        "Macro F1:",
        round(
            f1_score(actual, predictions, average="macro", zero_division=0),
            6,
        ),
    )

    print("\nPer-class report:")
    print(
        classification_report(
            actual,
            predictions,
            labels=list(multiclass_bundle["class_names"]),
            zero_division=0,
        )
    )

print("\nRecords changed to Bot:", int(np.sum(adjusted != original)))
print("Bot override applied:", int(bot_override.sum()))



Original hybrid
Accuracy: 0.998966
Balanced accuracy: 0.84694
Macro F1: 0.872575

Per-class report:
                          precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00    225274
                     Bot       0.86      0.74      0.80       188
                    DDoS       1.00      1.00      1.00     12886
           DoS GoldenEye       1.00      1.00      1.00      1059
                DoS Hulk       1.00      1.00      1.00     27448
        DoS Slowhttptest       0.99      0.99      0.99       549
           DoS slowloris       1.00      0.99      0.99       554
             FTP-Patator       1.00      1.00      1.00       716
              Heartbleed       0.00      0.00      0.00         0
            Infiltration       1.00      0.67      0.80         3
                PortScan       0.99      1.00      1.00     15627
             SSH-Patator       1.00      1.00      1.00       693
  Web Attack Brute Force       0.79     

In [22]:

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

actual = np.asarray(y_val_labels)
original = np.asarray(hybrid_val_predictions)
adjusted = np.asarray(adjusted)

assert len(actual) == len(original) == len(adjusted)
assert set(np.unique(actual)).issubset(
    set(multiclass_bundle["class_names"])
)

for name, predictions in [
    ("Original hybrid", original),
    ("Bot override", adjusted),
]:
    print(f"\n{name}")
    print("Records:", len(predictions))
    print("Accuracy:", accuracy_score(actual, predictions))
    print(
        "Balanced accuracy:",
        balanced_accuracy_score(actual, predictions),
    )
    print(
        "Macro-F1:",
        f1_score(
            actual,
            predictions,
            labels=list(multiclass_bundle["class_names"]),
            average="macro",
            zero_division=0,
        ),
    )

print("\nChanged predictions:", int(np.sum(original != adjusted)))
print("Actual label counts:")
print(
    __import__("pandas").Series(actual).value_counts()
)



Original hybrid
Records: 285201
Accuracy: 0.9989656417754496
Balanced accuracy: 0.8469395094440285
Macro-F1: 0.8144030066704556

Bot override
Records: 285201
Accuracy: 0.9989270724857205
Balanced accuracy: 0.8507322429832704
Macro-F1: 0.8135760107165979

Changed predictions: 31
Actual label counts:
BENIGN                      225274
DoS Hulk                     27448
PortScan                     15627
DDoS                         12886
DoS GoldenEye                 1059
FTP-Patator                    716
SSH-Patator                    693
DoS slowloris                  554
DoS Slowhttptest               549
Bot                            188
Web Attack Brute Force         150
Web Attack XSS                  51
Infiltration                     3
Web Attack SQL Injection         3
Name: count, dtype: int64


In [23]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

BINARY_BUNDLE_PATH = (
    PROJECT_ROOT / "models/binary/xgboost_binary_bundle.joblib"
)
MULTICLASS_BUNDLE_PATH = (
    PROJECT_ROOT / "models/multiclass/xgboost_multiclass_bundle.joblib"
)
TEST_FILE = PROJECT_ROOT / "data/splits/test.parquet"

assert BINARY_BUNDLE_PATH.exists(), f"Missing: {BINARY_BUNDLE_PATH}"
assert MULTICLASS_BUNDLE_PATH.exists(), f"Missing: {MULTICLASS_BUNDLE_PATH}"
assert TEST_FILE.exists(), f"Missing: {TEST_FILE}"

binary_bundle = joblib.load(BINARY_BUNDLE_PATH)
multiclass_bundle = joblib.load(MULTICLASS_BUNDLE_PATH)

binary_model = binary_bundle["model"]
binary_imputer = binary_bundle["imputer"]
binary_features = list(binary_bundle["feature_columns"])

multiclass_model = multiclass_bundle["model"]
multiclass_imputer = multiclass_bundle["imputer"]
multiclass_features = list(multiclass_bundle["feature_columns"])

CLASS_NAMES = list(multiclass_bundle["class_names"])
ID_TO_CLASS = {int(k): v for k, v in multiclass_bundle["id_to_class"].items()}

BINARY_THRESHOLD = 0.50

assert binary_features == multiclass_features, (
    "Binary and multiclass feature orders differ."
)
assert len(binary_features) == 78, (
    f"Expected 78 features, found {len(binary_features)}."
)
assert set(CLASS_NAMES) == set(ID_TO_CLASS.values())

test_columns = binary_features + ["label"]
test_df = pd.read_parquet(TEST_FILE, columns=test_columns)

X_test = test_df[binary_features].replace(
    [np.inf, -np.inf], np.nan
)
y_test = test_df["label"].astype(str).to_numpy()

assert len(X_test) == len(y_test)
assert set(np.unique(y_test)).issubset(set(CLASS_NAMES))

print("TEST EVALUATION PRECHECK")
print("=" * 55)
print("Test records:", f"{len(test_df):,}")
print("Feature count:", len(binary_features))
print("Binary threshold:", BINARY_THRESHOLD)
print("Binary and multiclass feature order matches: True")
print("True label alignment: passed")
print("\nTest label distribution:")
print(pd.Series(y_test).value_counts().to_string())


/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:45:59] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:45:59] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:45:59] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


TEST EVALUATION PRECHECK
Test records: 555,056
Feature count: 78
Binary threshold: 0.5
Binary and multiclass feature order matches: True
True label alignment: passed

Test label distribution:
BENIGN                      451830
DoS Hulk                     38087
PortScan                     32282
DDoS                         25538
DoS GoldenEye                 2058
FTP-Patator                   1318
DoS slowloris                 1106
DoS Slowhttptest              1058
SSH-Patator                    958
Bot                            371
Web Attack Brute Force         301
Web Attack XSS                 135
Infiltration                     9
Web Attack SQL Injection         3
Heartbleed                       2


In [24]:

import json
import time
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

# Preprocess using the saved imputers
X_binary_test = binary_imputer.transform(X_test)
X_multi_test = multiclass_imputer.transform(X_test)

assert np.isfinite(X_binary_test).all()
assert np.isfinite(X_multi_test).all()

# Binary stage
start_time = time.perf_counter()

binary_probabilities = binary_model.predict_proba(
    X_binary_test
)[:, 1]

binary_attack_mask = binary_probabilities >= BINARY_THRESHOLD

# Original hybrid pipeline, with no Bot override
hybrid_predictions = np.full(len(y_test), "BENIGN", dtype=object)

if binary_attack_mask.any():
    multiclass_ids = multiclass_model.predict(
        X_multi_test[binary_attack_mask]
    ).astype(np.int64)

    hybrid_predictions[binary_attack_mask] = [
        ID_TO_CLASS[int(class_id)]
        for class_id in multiclass_ids
    ]

elapsed_seconds = time.perf_counter() - start_time

# Evaluation
labels = CLASS_NAMES

accuracy = accuracy_score(y_test, hybrid_predictions)
balanced_acc = balanced_accuracy_score(y_test, hybrid_predictions)
macro_f1 = f1_score(
    y_test, hybrid_predictions,
    labels=labels, average="macro", zero_division=0
)
weighted_f1 = f1_score(
    y_test, hybrid_predictions,
    labels=labels, average="weighted", zero_division=0
)

report = classification_report(
    y_test,
    hybrid_predictions,
    labels=labels,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    y_test, hybrid_predictions, labels=labels
)

print("FINAL HYBRID TEST RESULTS")
print("=" * 65)
print("Test records:", f"{len(y_test):,}")
print("Binary threshold:", BINARY_THRESHOLD)
print("Routed to multiclass:", f"{int(binary_attack_mask.sum()):,}")
print("Kept as BENIGN by binary stage:", f"{int((~binary_attack_mask).sum()):,}")
print(f"Accuracy:          {accuracy:.6f}")
print(f"Balanced accuracy: {balanced_acc:.6f}")
print(f"Macro-F1:          {macro_f1:.6f}")
print(f"Weighted-F1:       {weighted_f1:.6f}")
print(f"Inference time:    {elapsed_seconds:.2f} seconds")
print(f"Throughput:        {len(y_test) / elapsed_seconds:.0f} records/second")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test, hybrid_predictions,
    labels=labels, zero_division=0, digits=5
))

print("\nCONFUSION MATRIX (rows=true, columns=predicted)")
display(pd.DataFrame(cm, index=labels, columns=labels))

# Save final baseline evaluation artifacts
metrics_dir = PROJECT_ROOT / "outputs" / "metrics"
predictions_dir = PROJECT_ROOT / "outputs" / "predictions"
metrics_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "evaluation": "original_hybrid_test",
    "test_records": int(len(y_test)),
    "binary_threshold": BINARY_THRESHOLD,
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "binary_routed_to_multiclass": int(binary_attack_mask.sum()),
    "binary_routed_as_benign": int((~binary_attack_mask).sum()),
    "inference_seconds": float(elapsed_seconds),
    "records_per_second": float(len(y_test) / elapsed_seconds),
    "classification_report": report,
    "class_names": labels,
    "confusion_matrix": cm.tolist(),
}

metrics_path = metrics_dir / "hybrid_final_test_metrics.json"
confusion_path = metrics_dir / "hybrid_final_test_confusion_matrix.csv"
predictions_path = predictions_dir / "hybrid_final_test_predictions.parquet"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame(cm, index=labels, columns=labels).to_csv(confusion_path)

pd.DataFrame({
    "true_label": y_test,
    "predicted_label": hybrid_predictions,
    "binary_attack_probability": binary_probabilities,
    "binary_prediction": np.where(
        binary_attack_mask, "ATTACK", "BENIGN"
    ),
}).to_parquet(predictions_path, index=False)

print("\nSaved metrics:", metrics_path)
print("Saved confusion matrix:", confusion_path)
print("Saved predictions:", predictions_path)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(


FINAL HYBRID TEST RESULTS
Test records: 555,056
Binary threshold: 0.5
Routed to multiclass: 103,392
Kept as BENIGN by binary stage: 451,664
Accuracy:          0.998897
Balanced accuracy: 0.812407
Macro-F1:          0.831224
Weighted-F1:       0.998856
Inference time:    11.07 seconds
Throughput:        50154 records/second

CLASSIFICATION REPORT
                          precision    recall  f1-score   support

                  BENIGN    0.99967   0.99934   0.99950    451830
                     Bot    0.97048   0.70889   0.81931       371
                    DDoS    0.99988   0.99984   0.99986     25538
           DoS GoldenEye    0.99708   0.99417   0.99562      2058
                DoS Hulk    0.99798   0.99950   0.99874     38087
        DoS Slowhttptest    0.99057   0.99244   0.99150      1058
           DoS slowloris    0.99637   0.99367   0.99502      1106
             FTP-Patator    1.00000   0.99772   0.99886      1318
              Heartbleed    1.00000   1.00000   1.00000  

,BENIGN,Bot,DDoS,DoS GoldenEye,DoS Hulk,DoS Slowhttptest,DoS slowloris,FTP-Patator,Heartbleed,Infiltration,PortScan,SSH-Patator,Web Attack Brute Force,Web Attack SQL Injection,Web Attack XSS
BENIGN,451533,8,2,4,60,7,1,0,0,0,213,0,2,0,0
Bot,108,263,0,0,0,0,0,0,0,0,0,0,0,0,0
DDoS,4,0,25534,0,0,0,0,0,0,0,0,0,0,0,0
DoS GoldenEye,3,0,0,2046,7,1,1,0,0,0,0,0,0,0,0
DoS Hulk,10,0,1,2,38068,0,0,0,0,0,5,0,1,0,0
DoS Slowhttptest,5,0,0,0,0,1050,2,0,0,0,0,0,1,0,0
DoS slowloris,3,0,0,0,1,2,1099,0,0,0,0,0,1,0,0
FTP-Patator,2,0,0,0,0,0,0,1315,0,0,0,0,0,0,1
Heartbleed,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0
Infiltration,6,0,0,0,0,0,0,0,0,3,0,0,0,0,0



Saved metrics: /content/drive/MyDrive/H-IDS/outputs/metrics/hybrid_final_test_metrics.json
Saved confusion matrix: /content/drive/MyDrive/H-IDS/outputs/metrics/hybrid_final_test_confusion_matrix.csv
Saved predictions: /content/drive/MyDrive/H-IDS/outputs/predictions/hybrid_final_test_predictions.parquet


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

METRICS_PATH = (
    PROJECT_ROOT / "outputs/metrics/hybrid_final_test_metrics.json"
)
PREDICTIONS_PATH = (
    PROJECT_ROOT / "outputs/predictions/hybrid_final_test_predictions.parquet"
)
TEST_PATH = PROJECT_ROOT / "data/splits/test.parquet"

# Mount Google Drive if it is not already mounted
from google.colab import drive
drive.mount("/content/drive")

# Verify the required artifacts
for path in [METRICS_PATH, PREDICTIONS_PATH, TEST_PATH]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

assert METRICS_PATH.exists(), "Final test metrics file is missing."
assert PREDICTIONS_PATH.exists(), "Prediction file is missing."
assert TEST_PATH.exists(), "Test split file is missing."

with open(METRICS_PATH, "r") as f:
    test_metrics = json.load(f)

predictions = pd.read_parquet(PREDICTIONS_PATH)

print("\nPrediction file shape:", predictions.shape)
print("Prediction columns:", predictions.columns.tolist())
print("\nSaved test metrics:")
print(json.dumps(test_metrics, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
hybrid_final_test_metrics.json: FOUND
hybrid_final_test_predictions.parquet: FOUND
test.parquet: FOUND

Prediction file shape: (555056, 4)
Prediction columns: ['true_label', 'predicted_label', 'binary_attack_probability', 'binary_prediction']

Saved test metrics:
{
  "evaluation": "original_hybrid_test",
  "test_records": 555056,
  "binary_threshold": 0.5,
  "accuracy": 0.9988974085497679,
  "balanced_accuracy": 0.8124066051007447,
  "macro_f1": 0.831223757372733,
  "weighted_f1": 0.9988558263762252,
  "binary_routed_to_multiclass": 103392,
  "binary_routed_as_benign": 451664,
  "inference_seconds": 11.06703307799944,
  "records_per_second": 50154.00207878805,
  "classification_report": {
    "BENIGN": {
      "precision": 0.9996656954862249,
      "recall": 0.9993426731292743,
      "f1-score": 0.9995041582089487,
      "support": 451830.0
    },
    "Bot": 

In [3]:
display(predictions.head())
print(predictions.dtypes)

,true_label,predicted_label,binary_attack_probability,binary_prediction
0,BENIGN,BENIGN,3.573966e-06,BENIGN
1,BENIGN,BENIGN,8.288955e-07,BENIGN
2,BENIGN,BENIGN,3.406847e-07,BENIGN
3,BENIGN,BENIGN,3.578696e-07,BENIGN
4,BENIGN,BENIGN,2.155481e-05,BENIGN


true_label                    object
predicted_label               object
binary_attack_probability    float32
binary_prediction             object
dtype: object


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
PREDICTIONS_PATH = (
    PROJECT_ROOT / "outputs/predictions/hybrid_final_test_predictions.parquet"
)
METRICS_DIR = PROJECT_ROOT / "outputs/metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

predictions = pd.read_parquet(PREDICTIONS_PATH)

required_cols = {
    "true_label",
    "predicted_label",
    "binary_attack_probability",
    "binary_prediction",
}
assert required_cols.issubset(predictions.columns), (
    f"Missing columns: {required_cols - set(predictions.columns)}"
)

# Validate the saved prediction records
assert predictions["true_label"].notna().all()
assert predictions["predicted_label"].notna().all()

predictions["correct"] = (
    predictions["true_label"] == predictions["predicted_label"]
)
predictions["is_actual_attack"] = predictions["true_label"] != "BENIGN"
predictions["is_predicted_attack"] = predictions["predicted_label"] != "BENIGN"

print("Total records:", len(predictions))
print("Correct predictions:", int(predictions["correct"].sum()))
print("Incorrect predictions:", int((~predictions["correct"]).sum()))
print("Actual attacks:", int(predictions["is_actual_attack"].sum()))
print("Predicted attacks:", int(predictions["is_predicted_attack"].sum()))

Total records: 555056
Correct predictions: 554444
Incorrect predictions: 612
Actual attacks: 103226
Predicted attacks: 103372


In [5]:
class_rows = []

for label, group in predictions.groupby("true_label", sort=False):
    support = len(group)
    correct = int((group["predicted_label"] == label).sum())
    missed_as_benign = int((group["predicted_label"] == "BENIGN").sum())
    misclassified = support - correct

    class_rows.append({
        "true_class": label,
        "support": support,
        "correct": correct,
        "recall_percent": round(100 * correct / support, 2),
        "missed_as_BENIGN": missed_as_benign,
        "other_misclassifications": misclassified - missed_as_benign,
        "total_errors": misclassified,
    })

class_errors = pd.DataFrame(class_rows).sort_values(
    ["recall_percent", "support"],
    ascending=[True, False]
)

display(class_errors)

class_errors.to_csv(
    METRICS_DIR / "hybrid_test_class_error_analysis.csv",
    index=False
)

print("Saved: hybrid_test_class_error_analysis.csv")

,true_class,support,correct,recall_percent,missed_as_BENIGN,other_misclassifications,total_errors
7,Web Attack SQL Injection,3,0,0.00,2,1,3
6,Web Attack XSS,135,41,30.37,2,92,94
4,Infiltration,9,3,33.33,6,0,6
3,Bot,371,263,70.89,108,0,108
5,Web Attack Brute Force,301,260,86.38,4,37,41
11,DoS Slowhttptest,1058,1050,99.24,5,3,8
10,DoS slowloris,1106,1099,99.37,3,4,7
13,DoS GoldenEye,2058,2046,99.42,3,9,12
8,FTP-Patator,1318,1315,99.77,2,1,3
0,BENIGN,451830,451533,99.93,451533,-451236,297


Saved: hybrid_test_class_error_analysis.csv


In [6]:
errors = predictions.loc[
    predictions["true_label"] != predictions["predicted_label"]
].copy()

confusion_pairs = (
    errors.groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(confusion_pairs.head(20))

confusion_pairs.to_csv(
    METRICS_DIR / "hybrid_test_top_confusion_pairs.csv",
    index=False
)

print("Total misclassified records:", len(errors))
print("Saved: hybrid_test_top_confusion_pairs.csv")

,true_label,predicted_label,count
6,BENIGN,PortScan,213
8,Bot,BENIGN,108
38,Web Attack XSS,Web Attack Brute Force,91
3,BENIGN,DoS Hulk,60
33,Web Attack Brute Force,Web Attack XSS,37
14,DoS Hulk,BENIGN,10
0,BENIGN,Bot,8
11,DoS GoldenEye,DoS Hulk,7
30,PortScan,DoS Hulk,7
4,BENIGN,DoS Slowhttptest,7


Total misclassified records: 612
Saved: hybrid_test_top_confusion_pairs.csv


In [7]:
# Stage 1 failure: an actual attack is classified as BENIGN
binary_missed_attacks = predictions[
    (predictions["true_label"] != "BENIGN") &
    (predictions["binary_prediction"] == "BENIGN")
].copy()

# Stage 2 failure: the binary stage routes a record as an attack,
# but the final multiclass prediction is incorrect.
multiclass_stage_errors = predictions[
    (predictions["binary_prediction"] != "BENIGN") &
    (predictions["true_label"] != predictions["predicted_label"])
].copy()

print("Actual attacks missed by binary detector:",
      len(binary_missed_attacks))
print("\nMissed attacks by true class:")
display(
    binary_missed_attacks["true_label"]
    .value_counts()
    .rename_axis("true_class")
    .reset_index(name="count")
)

print("\nErrors among records routed to multiclass:",
      len(multiclass_stage_errors))
print("\nMost frequent errors after routing:")
display(
    multiclass_stage_errors
    .groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(15)
)

binary_missed_attacks.to_parquet(
    PROJECT_ROOT / "outputs/predictions/hybrid_test_binary_missed_attacks.parquet",
    index=False
)

Actual attacks missed by binary detector: 142

Missed attacks by true class:


,true_class,count
0,Bot,108
1,DoS Hulk,7
2,Infiltration,6
3,DDoS,4
4,DoS Slowhttptest,4
5,Web Attack Brute Force,4
6,DoS slowloris,3
7,Web Attack XSS,2
8,PortScan,1
9,Web Attack SQL Injection,1



Errors among records routed to multiclass: 470

Most frequent errors after routing:


,true_label,predicted_label,count
6,BENIGN,PortScan,213
32,Web Attack XSS,Web Attack Brute Force,91
3,BENIGN,DoS Hulk,60
28,Web Attack Brute Force,Web Attack XSS,37
0,BENIGN,Bot,8
4,BENIGN,DoS Slowhttptest,7
26,PortScan,DoS Hulk,7
9,DoS GoldenEye,DoS Hulk,7
15,DoS Hulk,PortScan,5
2,BENIGN,DoS GoldenEye,4


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
PREDICTIONS_PATH = (
    PROJECT_ROOT / "outputs/predictions/hybrid_final_test_predictions.parquet"
)
METRICS_DIR = PROJECT_ROOT / "outputs/metrics"

predictions = pd.read_parquet(PREDICTIONS_PATH)

# 1. Binary-stage missed attacks
missed = predictions[
    (predictions["true_label"] != "BENIGN") &
    (predictions["binary_prediction"] == "BENIGN")
].copy()

print("BINARY-STAGE MISSED ATTACKS")
print("Total:", len(missed))

display(
    missed.groupby("true_label")
    .agg(
        count=("true_label", "size"),
        mean_attack_probability=("binary_attack_probability", "mean"),
        max_attack_probability=("binary_attack_probability", "max"),
    )
    .sort_values("count", ascending=False)
    .round(5)
)

# 2. Inspect the two web attack classes
web_errors = predictions[
    predictions["true_label"].isin([
        "Web Attack XSS",
        "Web Attack Brute Force",
    ])
].copy()

print("\nWEB ATTACK CLASSIFICATION")
display(
    pd.crosstab(
        web_errors["true_label"],
        web_errors["predicted_label"],
        margins=True
    )
)

# 3. Examine the binary probability distribution for Bot
bot = predictions[predictions["true_label"] == "Bot"].copy()

print("\nBOT BINARY PROBABILITY SUMMARY")
print(bot["binary_attack_probability"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
).round(5))

# 4. Save the targeted investigation data
missed.to_parquet(
    PROJECT_ROOT / "outputs/predictions/binary_missed_attack_analysis.parquet",
    index=False
)

web_errors.to_csv(
    METRICS_DIR / "web_attack_error_analysis.csv",
    index=False
)

print("\nSaved targeted analysis files.")

BINARY-STAGE MISSED ATTACKS
Total: 142


,count,mean_attack_probability,max_attack_probability
true_label,,,
Bot,108,0.35714,0.49939
DoS Hulk,7,0.32791,0.49766
Infiltration,6,0.03762,0.14359
DDoS,4,0.13439,0.28449
Web Attack Brute Force,4,0.25091,0.42146
DoS Slowhttptest,4,0.18207,0.29491
DoS slowloris,3,0.17148,0.45942
Web Attack XSS,2,0.18114,0.20967
DoS GoldenEye,1,0.45614,0.45614



WEB ATTACK CLASSIFICATION


predicted_label,BENIGN,DoS Hulk,Web Attack Brute Force,Web Attack XSS,All
true_label,,,,,
Web Attack Brute Force,4,0,260,37,301
Web Attack XSS,2,1,91,41,135
All,6,1,351,78,436



BOT BINARY PROBABILITY SUMMARY
count    371.00000
mean       0.77887
std        0.28762
min        0.00120
10%        0.35239
25%        0.45408
50%        0.96748
75%        0.99897
90%        0.99936
max        0.99957
Name: binary_attack_probability, dtype: float64

Saved targeted analysis files.


In [9]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

for folder in [
    PROJECT_ROOT / "data/splits",
    PROJECT_ROOT / "outputs/predictions",
    PROJECT_ROOT / "outputs/metrics",
]:
    print(f"\n{folder}")
    if folder.exists():
        for p in sorted(folder.iterdir()):
            print(" ", p.name)
    else:
        print("  Folder not found")


/content/drive/MyDrive/H-IDS/data/splits
  duplicate_aware_split.parquet
  duplicate_aware_split_summary.json
  test.parquet
  train.parquet
  validation.parquet

/content/drive/MyDrive/H-IDS/outputs/predictions
  binary_missed_attack_analysis.parquet
  hybrid_final_test_predictions.parquet
  hybrid_test_binary_missed_attacks.parquet

/content/drive/MyDrive/H-IDS/outputs/metrics
  binary_test_metrics.json
  binary_validation_metrics.json
  hybrid_final_test_confusion_matrix.csv
  hybrid_final_test_metrics.json
  hybrid_test_class_error_analysis.csv
  hybrid_test_metrics.json
  hybrid_test_top_confusion_pairs.csv
  web_attack_error_analysis.csv


In [10]:
from pathlib import Path
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

metrics_path = (
    PROJECT_ROOT / "outputs/metrics/binary_validation_metrics.json"
)

with open(metrics_path, "r") as f:
    binary_val_metrics = json.load(f)

print(json.dumps(binary_val_metrics, indent=2))

{
  "validation_metrics": {
    "accuracy": 0.9992426394016851,
    "balanced_accuracy": 0.999098030221359,
    "attack_precision": 0.9975502041496542,
    "attack_recall": 0.9988485991289402,
    "attack_f1": 0.9981989794216722,
    "average_precision": 0.9999040172810715,
    "confusion_matrix": [
      [
        225127,
        147
      ],
      [
        69,
        59858
      ]
    ]
  },
  "best_iteration": 253,
  "decision_threshold": 0.5,
  "created_utc": "2026-10-14T14:36:47.863186+00:00"
}


In [11]:
for folder in [
    PROJECT_ROOT / "models/binary",
    PROJECT_ROOT / "models/multiclass",
]:
    print(f"\n{folder}")
    if folder.exists():
        for p in sorted(folder.iterdir()):
            print(" ", p.name)
    else:
        print("Folder not found")


/content/drive/MyDrive/H-IDS/models/binary
  xgboost_binary_bundle.joblib

/content/drive/MyDrive/H-IDS/models/multiclass
  multiclass_label_mapping.json
  xgboost_multiclass_bundle.joblib
  xgboost_multiclass_model.json


In [12]:
from pathlib import Path
import joblib
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

BINARY_BUNDLE_PATH = (
    PROJECT_ROOT / "models/binary/xgboost_binary_bundle.joblib"
)
MULTICLASS_BUNDLE_PATH = (
    PROJECT_ROOT / "models/multiclass/xgboost_multiclass_bundle.joblib"
)
VALIDATION_PATH = PROJECT_ROOT / "data/splits/validation.parquet"

for path in [
    BINARY_BUNDLE_PATH,
    MULTICLASS_BUNDLE_PATH,
    VALIDATION_PATH,
]:
    print(path.name, "FOUND" if path.exists() else "MISSING")
    assert path.exists(), f"Required file missing: {path}"

binary_bundle = joblib.load(BINARY_BUNDLE_PATH)
multiclass_bundle = joblib.load(MULTICLASS_BUNDLE_PATH)

print("\nBinary bundle type:", type(binary_bundle))
print("Multiclass bundle type:", type(multiclass_bundle))

if isinstance(binary_bundle, dict):
    print("\nBinary bundle keys:", list(binary_bundle.keys()))

if isinstance(multiclass_bundle, dict):
    print("Multiclass bundle keys:", list(multiclass_bundle.keys()))

xgboost_binary_bundle.joblib FOUND
xgboost_multiclass_bundle.joblib FOUND
validation.parquet FOUND


/usr/lib/python3.13/pickle.py:1754: UserWarning: [15:33:09] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [15:33:09] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [15:33:09] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)



Binary bundle type: <class 'dict'>
Multiclass bundle type: <class 'dict'>

Binary bundle keys: ['model', 'imputer', 'feature_columns', 'label_mapping', 'decision_threshold', 'best_iteration', 'validation_metrics', 'created_utc', 'xgboost_version']
Multiclass bundle keys: ['model', 'imputer', 'feature_columns', 'class_names', 'class_to_id', 'id_to_class', 'best_iteration', 'num_classes', 'model_type']


/usr/lib/python3.13/pickle.py:1754: UserWarning: [15:33:11] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [15:33:11] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [15:33:11] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


In [13]:
import pandas as pd
import numpy as np

validation = pd.read_parquet(VALIDATION_PATH)

binary_features = binary_bundle["feature_columns"]
multiclass_features = multiclass_bundle["feature_columns"]

print("Validation shape:", validation.shape)
print("Validation columns:", validation.columns.tolist())
print("\nBinary feature count:", len(binary_features))
print("Multiclass feature count:", len(multiclass_features))
print("Feature orders identical:", binary_features == multiclass_features)

assert binary_features == multiclass_features, (
    "Binary and multiclass feature orders differ."
)

print("\nBinary label mapping:", binary_bundle["label_mapping"])
print("Multiclass class names:", multiclass_bundle["class_names"])

Validation shape: (285201, 82)
Validation columns: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 

In [14]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    precision_score,
)
import time

FEATURES = binary_bundle["feature_columns"]
LABEL_COL = "label"

assert LABEL_COL in validation.columns, (
    f"Expected '{LABEL_COL}' in validation.parquet. "
    f"Found: {validation.columns.tolist()}"
)

# Ensure the same feature order used during training
X_val = validation[FEATURES].copy()
y_true = validation[LABEL_COL].astype(str).to_numpy()

# Apply saved imputers
X_binary = binary_bundle["imputer"].transform(X_val)
X_multi = multiclass_bundle["imputer"].transform(X_val)

binary_model = binary_bundle["model"]
multiclass_model = multiclass_bundle["model"]

# Compute probabilities once, then evaluate multiple thresholds
binary_prob = binary_model.predict_proba(X_binary)[:, 1]
multi_prob = multiclass_model.predict_proba(X_multi)

class_names = multiclass_bundle["class_names"]
multi_pred_ids = np.argmax(multi_prob, axis=1)
multi_pred_labels = np.asarray(class_names)[multi_pred_ids]

print("Validation records:", len(y_true))
print("Binary probability shape:", binary_prob.shape)
print("Multiclass probability shape:", multi_prob.shape)
print("Class count:", len(class_names))
print("Binary model classes:", binary_model.classes_)
print("Saved binary label mapping:", binary_bundle["label_mapping"])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(


Validation records: 285201
Binary probability shape: (285201,)
Multiclass probability shape: (285201, 15)
Class count: 15
Binary model classes: [0 1]
Saved binary label mapping: {'BENIGN': 0, 'ATTACK': 1}


In [15]:
# Verify that probability column 1 corresponds to ATTACK.
# This is important before interpreting threshold results.
label_mapping = binary_bundle["label_mapping"]
print("Binary label mapping:", label_mapping)

# The known binary training convention is BENIGN=0, ATTACK=1.
assert np.array_equal(
    np.asarray(binary_model.classes_), np.array([0, 1])
), f"Unexpected binary classes: {binary_model.classes_}"

thresholds = [0.50, 0.40, 0.30, 0.20]
results = []

for threshold in thresholds:
    binary_attack = binary_prob >= threshold

    # Start all final predictions as BENIGN.
    y_pred = np.full(len(y_true), "BENIGN", dtype=object)

    # Only records routed by the binary stage receive multiclass labels.
    y_pred[binary_attack] = multi_pred_labels[binary_attack]

    actual_attack = y_true != "BENIGN"
    predicted_attack = y_pred != "BENIGN"

    results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "attack_precision": precision_score(
            actual_attack, predicted_attack, zero_division=0
        ),
        "attack_recall": recall_score(
            actual_attack, predicted_attack, zero_division=0
        ),
        "missed_attacks": int((actual_attack & ~predicted_attack).sum()),
        "false_alarms": int((~actual_attack & predicted_attack).sum()),
        "routed_to_multiclass": int(binary_attack.sum()),
        "bot_recall": recall_score(
            y_true == "Bot", y_pred == "Bot", zero_division=0
        ),
    })

threshold_results = pd.DataFrame(results)

display(
    threshold_results.style.format({
        "accuracy": "{:.6f}",
        "balanced_accuracy": "{:.6f}",
        "macro_f1": "{:.6f}",
        "attack_precision": "{:.6f}",
        "attack_recall": "{:.6f}",
        "bot_recall": "{:.6f}",
    })
)

threshold_results.to_csv(
    PROJECT_ROOT / "outputs/metrics/hybrid_validation_threshold_comparison.csv",
    index=False
)

print("Saved validation threshold comparison.")

Binary label mapping: {'BENIGN': 0, 'ATTACK': 1}


,threshold,accuracy,balanced_accuracy,macro_f1,attack_precision,attack_recall,missed_attacks,false_alarms,routed_to_multiclass,bot_recall
0,0.500000,0.998983,0.843053,0.873121,0.997650,0.998715,77,141,60005,0.696809
1,0.400000,0.998973,0.846458,0.872934,0.997401,0.998915,65,156,60075,0.739362
2,0.300000,0.998966,0.846940,0.872575,0.997334,0.998949,63,160,60142,0.744681
3,0.200000,0.998966,0.846940,0.872575,0.997334,0.998949,63,160,60172,0.744681


Saved validation threshold comparison.


In [16]:
# Display the complete threshold comparison
display(
    threshold_results[
        [
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "attack_precision",
            "attack_recall",
            "missed_attacks",
            "false_alarms",
            "routed_to_multiclass",
            "bot_recall",
        ]
    ].round(6)
    .sort_values("threshold", ascending=False)
    .reset_index(drop=True)
)

,threshold,accuracy,balanced_accuracy,macro_f1,attack_precision,attack_recall,missed_attacks,false_alarms,routed_to_multiclass,bot_recall
0,0.5,0.998983,0.843053,0.873121,0.997650,0.998715,77,141,60005,0.696809
1,0.4,0.998973,0.846458,0.872934,0.997401,0.998915,65,156,60075,0.739362
2,0.3,0.998966,0.846940,0.872575,0.997334,0.998949,63,160,60142,0.744681
3,0.2,0.998966,0.846940,0.872575,0.997334,0.998949,63,160,60172,0.744681


In [17]:
bot_mask = y_true == "Bot"

for threshold in [0.50, 0.40, 0.30, 0.20]:
    binary_attack = binary_prob >= threshold
    final_pred = np.full(len(y_true), "BENIGN", dtype=object)
    final_pred[binary_attack] = multi_pred_labels[binary_attack]

    bot_total = int(bot_mask.sum())
    bot_correct = int(((final_pred == "Bot") & bot_mask).sum())
    bot_missed_by_binary = int(
        (bot_mask & ~binary_attack).sum()
    )

    print(
        f"Threshold={threshold:.2f} | "
        f"Bot correctly detected={bot_correct}/{bot_total} | "
        f"Bot recall={bot_correct / bot_total:.2%} | "
        f"Bot missed by binary stage={bot_missed_by_binary}"
    )

Threshold=0.50 | Bot correctly detected=131/188 | Bot recall=69.68% | Bot missed by binary stage=57
Threshold=0.40 | Bot correctly detected=139/188 | Bot recall=73.94% | Bot missed by binary stage=34
Threshold=0.30 | Bot correctly detected=140/188 | Bot recall=74.47% | Bot missed by binary stage=13
Threshold=0.20 | Bot correctly detected=140/188 | Bot recall=74.47% | Bot missed by binary stage=4


In [18]:
import pandas as pd
import numpy as np

THRESHOLD = 0.30
BOT_CLASS = "Bot"

binary_attack = binary_prob >= THRESHOLD

# Original hybrid predictions
hybrid_pred = np.full(len(y_true), "BENIGN", dtype=object)
hybrid_pred[binary_attack] = multi_pred_labels[binary_attack]

# Actual Bot records
bot_mask = y_true == BOT_CLASS

# Bot records routed to multiclass but not predicted as Bot
bot_routed_wrong = bot_mask & binary_attack & (hybrid_pred != BOT_CLASS)

# Bot records that never reach multiclass
bot_missed_stage1 = bot_mask & ~binary_attack

print("Validation Bot records:", int(bot_mask.sum()))
print("Bot records missed by binary stage:", int(bot_missed_stage1.sum()))
print("Bot records routed but misclassified by multiclass:",
      int(bot_routed_wrong.sum()))
print("Bot records correctly predicted end-to-end:",
      int((bot_mask & (hybrid_pred == BOT_CLASS)).sum()))

# Inspect multiclass probabilities for the routed Bot errors
bot_error_indices = np.flatnonzero(bot_routed_wrong)
bot_class_id = list(class_names).index(BOT_CLASS)

bot_error_report = pd.DataFrame({
    "true_label": y_true[bot_error_indices],
    "predicted_label": hybrid_pred[bot_error_indices],
    "binary_attack_probability": binary_prob[bot_error_indices],
    "multiclass_bot_probability": multi_prob[bot_error_indices, bot_class_id],
    "multiclass_predicted_probability": multi_prob[
        bot_error_indices,
        multi_pred_ids[bot_error_indices]
    ],
})

display(bot_error_report.head(20))

print("\nMulticlass Bot probability summary for routed Bot errors:")
display(
    bot_error_report["multiclass_bot_probability"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .to_frame()
)

Validation Bot records: 188
Bot records missed by binary stage: 13
Bot records routed but misclassified by multiclass: 35
Bot records correctly predicted end-to-end: 140


,true_label,predicted_label,binary_attack_probability,multiclass_bot_probability,multiclass_predicted_probability
0,Bot,BENIGN,0.352691,0.319553,0.679871
1,Bot,BENIGN,0.343774,0.280168,0.719230
2,Bot,BENIGN,0.440541,0.350712,0.648727
3,Bot,BENIGN,0.438711,0.437963,0.561571
4,Bot,BENIGN,0.394827,0.320204,0.679211
5,Bot,BENIGN,0.315090,0.307105,0.692333
6,Bot,BENIGN,0.393995,0.464036,0.535530
7,Bot,BENIGN,0.309382,0.201206,0.798157
8,Bot,BENIGN,0.459292,0.421918,0.577580
9,Bot,BENIGN,0.350172,0.201259,0.798086



Multiclass Bot probability summary for routed Bot errors:


,multiclass_bot_probability
count,35.000000
mean,0.343972
std,0.087421
min,0.143695
25%,0.291665
50%,0.350712
75%,0.410605
90%,0.452713
max,0.495897


In [19]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
import pandas as pd
import numpy as np

BINARY_THRESHOLD = 0.30
BOT_THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.49]

binary_attack = binary_prob >= BINARY_THRESHOLD

# Baseline hybrid predictions at binary threshold 0.30
baseline_pred = np.full(len(y_true), "BENIGN", dtype=object)
baseline_pred[binary_attack] = multi_pred_labels[binary_attack]

bot_class_id = list(class_names).index("Bot")
bot_probability = multi_prob[:, bot_class_id]

actual_bot = y_true == "Bot"
results = []

for bot_threshold in BOT_THRESHOLDS:
    # Start with the original hybrid predictions
    candidate_pred = baseline_pred.copy()

    # Override only records routed to multiclass
    override_mask = (
        binary_attack &
        (bot_probability >= bot_threshold)
    )
    candidate_pred[override_mask] = "Bot"

    bot_tp = int(((candidate_pred == "Bot") & actual_bot).sum())
    bot_fp = int(((candidate_pred == "Bot") & ~actual_bot).sum())
    bot_fn = int(((candidate_pred != "Bot") & actual_bot).sum())

    results.append({
        "bot_threshold": bot_threshold,
        "accuracy": accuracy_score(y_true, candidate_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, candidate_pred),
        "macro_f1": f1_score(
            y_true, candidate_pred, average="macro", zero_division=0
        ),
        "bot_precision": precision_score(
            actual_bot, candidate_pred == "Bot", zero_division=0
        ),
        "bot_recall": recall_score(
            actual_bot, candidate_pred == "Bot", zero_division=0
        ),
        "bot_true_positives": bot_tp,
        "bot_false_positives": bot_fp,
        "bot_false_negatives": bot_fn,
        "changed_predictions": int((candidate_pred != baseline_pred).sum()),
    })

bot_override_results = pd.DataFrame(results)

display(
    bot_override_results.style.format({
        "accuracy": "{:.6f}",
        "balanced_accuracy": "{:.6f}",
        "macro_f1": "{:.6f}",
        "bot_precision": "{:.4f}",
        "bot_recall": "{:.4f}",
    })
)

bot_override_results.to_csv(
    PROJECT_ROOT / "outputs/metrics/bot_override_validation_comparison.csv",
    index=False,
)

print("Saved Bot override validation comparison.")

,bot_threshold,accuracy,balanced_accuracy,macro_f1,bot_precision,bot_recall,bot_true_positives,bot_false_positives,bot_false_negatives,changed_predictions
0,0.300000,0.998924,0.856806,0.872736,0.7313,0.8830,166,61,22,64
1,0.350000,0.998931,0.853770,0.872448,0.7560,0.8404,158,51,30,46
2,0.400000,0.998927,0.850732,0.871689,0.7732,0.7979,150,44,38,31
3,0.450000,0.998962,0.848458,0.872737,0.8372,0.7660,144,28,44,9
4,0.490000,0.998969,0.847319,0.872819,0.8598,0.7500,141,23,47,1


Saved Bot override validation comparison.


In [20]:
import pandas as pd
import numpy as np

BINARY_THRESHOLD = 0.30
BOT_THRESHOLD = 0.30

binary_attack = binary_prob >= BINARY_THRESHOLD

# Reconstruct the original hybrid predictions
baseline_pred = np.full(len(y_true), "BENIGN", dtype=object)
baseline_pred[binary_attack] = multi_pred_labels[binary_attack]

bot_class_id = list(class_names).index("Bot")
bot_probability = multi_prob[:, bot_class_id]

# Apply the experimental override
override_pred = baseline_pred.copy()
override_mask = binary_attack & (bot_probability >= BOT_THRESHOLD)
override_pred[override_mask] = "Bot"

# Newly predicted Bot records that are not actually Bot
false_bot_mask = (override_pred == "Bot") & (y_true != "Bot")

false_bot_analysis = pd.DataFrame({
    "true_label": y_true[false_bot_mask],
    "original_prediction": baseline_pred[false_bot_mask],
    "binary_attack_probability": binary_prob[false_bot_mask],
    "multiclass_bot_probability": bot_probability[false_bot_mask],
})

print("False Bot predictions:", len(false_bot_analysis))
print("\nBreakdown by actual class:")

display(
    false_bot_analysis["true_label"]
    .value_counts()
    .rename_axis("actual_class")
    .reset_index(name="false_bot_count")
)

print("\nExamples of false Bot predictions:")
display(false_bot_analysis.head(20))

# Save the analysis
false_bot_analysis.to_csv(
    PROJECT_ROOT / "outputs/metrics/bot_override_false_positive_analysis.csv",
    index=False,
)

print("\nSaved false-positive analysis.")

False Bot predictions: 61

Breakdown by actual class:


,actual_class,false_bot_count
0,BENIGN,61



Examples of false Bot predictions:


,true_label,original_prediction,binary_attack_probability,multiclass_bot_probability
0,BENIGN,BENIGN,0.350695,0.349851
1,BENIGN,BENIGN,0.409113,0.487379
2,BENIGN,Bot,0.352884,0.657905
3,BENIGN,BENIGN,0.441012,0.435816
4,BENIGN,BENIGN,0.409113,0.481611
5,BENIGN,Bot,0.476332,0.538311
6,BENIGN,Bot,0.391489,0.513038
7,BENIGN,BENIGN,0.438952,0.417703
8,BENIGN,BENIGN,0.417726,0.450117
9,BENIGN,BENIGN,0.332637,0.300627



Saved false-positive analysis.


In [21]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
import pandas as pd
import numpy as np

binary_thresholds = [0.30, 0.35, 0.40]
bot_thresholds = [0.30, 0.35, 0.40, 0.45]

bot_class_id = list(class_names).index("Bot")
bot_probability = multi_prob[:, bot_class_id]
actual_bot = y_true == "Bot"
actual_attack = y_true != "BENIGN"

results = []

for binary_threshold in binary_thresholds:
    binary_attack = binary_prob >= binary_threshold

    # Original hybrid prediction for this binary threshold
    base_pred = np.full(len(y_true), "BENIGN", dtype=object)
    base_pred[binary_attack] = multi_pred_labels[binary_attack]

    for bot_threshold in bot_thresholds:
        candidate_pred = base_pred.copy()

        override_mask = (
            binary_attack &
            (bot_probability >= bot_threshold)
        )
        candidate_pred[override_mask] = "Bot"

        predicted_attack = candidate_pred != "BENIGN"

        results.append({
            "binary_threshold": binary_threshold,
            "bot_threshold": bot_threshold,
            "accuracy": accuracy_score(y_true, candidate_pred),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, candidate_pred
            ),
            "macro_f1": f1_score(
                y_true, candidate_pred,
                average="macro", zero_division=0
            ),
            "attack_recall": recall_score(
                actual_attack, predicted_attack, zero_division=0
            ),
            "bot_precision": precision_score(
                actual_bot, candidate_pred == "Bot",
                zero_division=0
            ),
            "bot_recall": recall_score(
                actual_bot, candidate_pred == "Bot",
                zero_division=0
            ),
            "false_alarms": int(
                ((y_true == "BENIGN") & predicted_attack).sum()
            ),
            "bot_false_positives": int(
                ((y_true != "Bot") & (candidate_pred == "Bot")).sum()
            ),
        })

grid_results = pd.DataFrame(results)

display(
    grid_results.sort_values(
        ["balanced_accuracy", "macro_f1"],
        ascending=False
    ).round(6)
)

grid_results.to_csv(
    PROJECT_ROOT / "outputs/metrics/bot_override_threshold_grid.csv",
    index=False
)

print("Saved two-threshold validation experiment.")

,binary_threshold,bot_threshold,accuracy,balanced_accuracy,macro_f1,attack_recall,bot_precision,bot_recall,false_alarms,bot_false_positives
0,0.30,0.30,0.998924,0.856806,0.872736,0.999383,0.731278,0.882979,198,61
4,0.35,0.30,0.998934,0.856048,0.873016,0.999349,0.745455,0.872340,193,56
1,0.30,0.35,0.998931,0.853770,0.872448,0.999249,0.755981,0.840426,188,51
5,0.35,0.35,0.998938,0.853391,0.872663,0.999232,0.765854,0.835106,185,48
8,0.40,0.30,0.998959,0.851771,0.873366,0.999149,0.805263,0.813830,174,37
9,0.40,0.35,0.998959,0.851012,0.873220,0.999116,0.811828,0.803191,172,35
2,0.30,0.40,0.998927,0.850732,0.871689,0.999116,0.773196,0.797872,181,44
6,0.35,0.40,0.998931,0.850353,0.871756,0.999099,0.780105,0.792553,179,42
10,0.40,0.40,0.998948,0.849113,0.872374,0.999032,0.815642,0.776596,170,33
3,0.30,0.45,0.998962,0.848458,0.872737,0.999015,0.837209,0.765957,165,28


Saved two-threshold validation experiment.


In [22]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

split_paths = {
    "train": PROJECT_ROOT / "data/splits/train.parquet",
    "validation": PROJECT_ROOT / "data/splits/validation.parquet",
    "test": PROJECT_ROOT / "data/splits/test.parquet",
}

target_classes = [
    "BENIGN",
    "Bot",
    "Web Attack Brute Force",
    "Web Attack SQL Injection",
    "Web Attack XSS",
    "Infiltration",
]

rows = []

for split_name, path in split_paths.items():
    df = pd.read_parquet(path, columns=["label"])
    counts = df["label"].value_counts()

    for label in target_classes:
        rows.append({
            "split": split_name,
            "class": label,
            "count": int(counts.get(label, 0)),
        })

class_distribution = pd.DataFrame(rows)

display(
    class_distribution.pivot(
        index="class",
        columns="split",
        values="count"
    ).fillna(0).astype(int)
)

class_distribution.to_csv(
    PROJECT_ROOT / "outputs/metrics/target_class_distribution.csv",
    index=False
)

print("Saved target class distribution.")

split,test,train,validation
class,,,
BENIGN,451830,1595993,225274
Bot,371,1407,188
Infiltration,9,24,3
Web Attack Brute Force,301,1056,150
Web Attack SQL Injection,3,15,3
Web Attack XSS,135,466,51


Saved target class distribution.


In [23]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")

for folder_name in ["src", "notebooks"]:
    folder = PROJECT_ROOT / folder_name
    print(f"\n--- {folder_name} ---")

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in {
                ".py", ".ipynb"
            }:
                print(path.relative_to(PROJECT_ROOT))
    else:
        print("Folder not found")


--- src ---

--- notebooks ---
notebooks/00_environment_setup.ipynb
notebooks/01_binary_detector.ipynb
notebooks/02_multiclass_detector.ipynb
notebooks/03_hybrid_inference.ipynb


In [24]:
from pathlib import Path
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/H-IDS")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/02_multiclass_detector.ipynb"

assert NOTEBOOK_PATH.exists(), f"Notebook not found: {NOTEBOOK_PATH}"

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Notebook:", NOTEBOOK_PATH.name)
print("Total cells:", len(notebook["cells"]))

keywords = [
    "XGBClassifier",
    "xgb.train",
    "objective",
    "fit(",
    "class_weight",
    "sample_weight",
    "best_iteration",
    "predict_proba",
]

for i, cell in enumerate(notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword.lower() in source.lower() for keyword in keywords):
        print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
        print(source[:6000])


Notebook: 02_multiclass_detector.ipynb
Total cells: 21

========================= CELL 9 =========================

from xgboost import XGBClassifier
import time

MULTICLASS_MODEL_PATH = (
    MULTICLASS_MODEL_DIR / "xgboost_multiclass_model.json"
)

NUM_CLASSES = len(CLASS_NAMES)

multiclass_model = XGBClassifier(
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    n_estimators=500,
    learning_rate=0.08,
    max_depth=8,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=4,
    early_stopping_rounds=30,
)

print("Multiclass model configured.")
print("Number of classes:", NUM_CLASSES)
print("Training records:", f"{len(y_train):,}")
print("Validation records:", f"{len(y_val):,}")
print("GPU requested: CUDA")


========================= CELL 15 =========================

import xgboost as xgb
import numpy as np
import time

NUM_CLASSES 